# 01 Baseline Evaluation and Dataset Preparation

> Nemotron SFT-RL 课程大作业 · 第 1 个 Notebook
>
> 本 Notebook 是为 `datawhalechina/llm-algo-leetcode` 后训练优化路线准备的参考实现。它先固定真实数据身份、分类、划分和泄漏审计，再提供一个默认关闭的 Qwen3/vLLM Baseline 入口，以及可选的数据合成与筛查 Pilot。

**当前证据状态**：数据读取、分类、清洗、IID 划分、审计和合成 smoke test 可在无模型下载条件下验证；Qwen3 模型权重和 vLLM GPU 推理只有在目标 RTX 5090 与官方 vLLM Profile 准备好后才运行。Notebook 不在执行过程中安装依赖。模型路径通过 `QWEN3_MODEL_PATH` 环境变量提供。Notebook 不在执行过程中下载权重。


## 学习目标

完成本节后，你应当能够：

1. 从官方 `train.csv` 的实际样本归纳可审计的题型分类，而不是依赖旧统计。
2. 将 Equation Numeric 与 Equation Symbolic 分开，并保留所有被排除样本的审计记录。
3. 在固定随机种子下生成 Train / Validation / Holdout Test，并检查精确重复、规范化重复和跨集合污染。
4. 理解原始模型 Baseline 的边界：原始 Prompt、官方 Chat Template、统一解码和确定性答案验证。
5. 在已确认的先验范围内构造小型合成样本；Equation Numeric 使用明确的 24 条规则，Guess 只有在全局约束下答案唯一时才进入合格数据。


## 1. 实验约束与运行配置

首版只研究三个真实题型：Gravity、Equation Numeric、Bit Manipulation。Gravity 是能力保持控制组，Equation Numeric 和 Bit Manipulation 是后续 RL 的主要任务。模型固定为 `Qwen/Qwen3-4B-Thinking-2507`，训练和 vLLM Rollout 使用 Datawhale 官方隔离 Profile。

`RUN_BASELINE=False` 是安全默认值：它不会下载模型或占用 GPU。准备好官方 vLLM Profile 后，将其改为 `True`，并先运行 `smoke`。模型路径通过 `LOCAL_MODEL_PATH` 环境变量配置；本 Notebook 不保存机器专属路径。


In [ ]:
from __future__ import annotations

import hashlib
import os
import json
import math
import re
import time
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_CANDIDATES = [
    PROJECT_ROOT / "train.csv",
    PROJECT_ROOT.parent / "train.csv",
]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError(f"未找到 train.csv，检查路径：{RAW_CANDIDATES}")

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = OUTPUT_DIR / "splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
RUN_MODE = os.environ.get("RUN_MODE", "smoke")                 # smoke / full
MAX_SAMPLES_PER_TASK = int(os.environ.get("MAX_SAMPLES_PER_TASK", "3"))
SEED = 42
RUN_BASELINE = os.environ.get("RUN_BASELINE", "0") == "1"   # 默认不下载模型
RUN_SYNTHETIC = os.environ.get("RUN_SYNTHETIC", "0") == "1"  # 默认复用已有 Pilot，不重复生成
SYNTHETIC_SAMPLES_PER_TASK = 100
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "2048"))
VLLM_MAX_MODEL_LEN = int(os.environ.get("VLLM_MAX_MODEL_LEN", "8192"))
SOURCE_DATASET = "NVIDIA Nemotron Model Reasoning Challenge / train.csv"

TARGET_TASKS = {"gravity", "equation_numeric", "bit_manipulation"}
print({"raw_path": str(RAW_PATH), "output_dir": str(OUTPUT_DIR), "model_id": MODEL_ID, "run_baseline": RUN_BASELINE})


## 2. 读取并核验官方数据

这里使用字符串读取，避免 ID、二进制答案或带前导零的数值被自动转换。`source_row` 是原始 CSV 中的行号（表头为第 1 行），后续所有产物都保留它和原始 `id`。


In [ ]:
raw_df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
required_columns = ["id", "prompt", "answer"]
missing_columns = [c for c in required_columns if c not in raw_df.columns]
if missing_columns:
    raise ValueError(f"缺少字段：{missing_columns}")
raw_df = raw_df[required_columns].copy()
raw_df.insert(0, "source_row", np.arange(2, len(raw_df) + 2, dtype=int))
raw_df["source_dataset"] = SOURCE_DATASET

empty_counts = {c: int((raw_df[c].str.len() == 0).sum()) for c in required_columns}
raw_profile = {
    "sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "rows": int(len(raw_df)),
    "columns": list(raw_df.columns),
    "dtypes": {c: str(raw_df[c].dtype) for c in raw_df.columns},
    "empty_counts": empty_counts,
    "null_counts": {c: int(raw_df[c].isna().sum()) for c in raw_df.columns},
    "duplicate_id_rows": int(raw_df["id"].duplicated(keep=False).sum()),
    "duplicate_id_repeats": int(raw_df["id"].duplicated().sum()),
    "duplicate_prompt_rows": int(raw_df["prompt"].duplicated(keep=False).sum()),
    "duplicate_prompt_repeats": int(raw_df["prompt"].duplicated().sum()),
}
print(json.dumps(raw_profile, ensure_ascii=False, indent=2))
print("\n首行字段：", raw_df.iloc[0][required_columns].to_dict())


## 3. 题型分类：使用精确模板，再细分 Equation

官方数据的六个首句模板可以被精确识别：Gravity、Bit Manipulation、Equation、Unit Conversion、Encryption 和 Numeral System。Equation 模板内部必须再分：含数字的样本是 Equation Numeric，完全由符号构成的样本是 Equation Symbolic。

这里不使用宽泛的 `"equation" in prompt` 规则。遇到没有匹配模板的样本会进入 `unknown`，不会静默丢弃。Numeric 样本还会根据 Query 运算符是否在示例中出现，标记 `deduce` 或 `guess`。


In [ ]:
TEMPLATE_PREFIXES = [
    ("gravity", "In Alice's Wonderland, the gravitational constant has been secretly changed.", "exact_gravity_prefix"),
    ("bit_manipulation", "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.", "exact_bit_prefix"),
    ("equation", "In Alice's Wonderland, a secret set of transformation rules is applied to equations.", "exact_equation_prefix"),
    ("unit_conversion", "In Alice's Wonderland, a secret unit conversion is applied to measurements.", "exact_unit_prefix"),
    ("encryption", "In Alice's Wonderland, secret encryption rules are used on text.", "exact_encryption_prefix"),
    ("numeral_system", "In Alice's Wonderland, numbers are secretly converted into a different numeral system.", "exact_numeral_prefix"),
]


def normalize_prompt(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"0b[01]+", "<bin>", text)
    text = re.sub(r"[-+]?\d+(?:\.\d+)?", "<num>", text)
    text = re.sub(r"\s+", " ", text)
    return text


def prompt_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def query_operator_and_example_operators(prompt: str) -> tuple[str | None, set[str]]:
    example_ops = set(re.findall(r"(?m)^\s*\d+\s*(.)\s*\d+\s*=", prompt))
    query = re.search(r"Now, determine the result for:\s*([0-9]+)\s*(.)\s*([0-9]+)", prompt)
    if query is None:
        return None, example_ops
    return query.group(2), example_ops


def classify_prompt(prompt: str) -> dict[str, object]:
    prompt = str(prompt)
    for base_label, prefix, rule in TEMPLATE_PREFIXES:
        if prompt.startswith(prefix):
            if base_label == "equation":
                if re.search(r"\d", prompt):
                    query_op, example_ops = query_operator_and_example_operators(prompt)
                    subtype = "deduce" if query_op in example_ops else "guess"
                    if query_op is None:
                        subtype = "undetermined"
                    return {
                        "task_type": "equation_numeric",
                        "equation_subtype": subtype,
                        "classification_confidence": "high",
                        "classification_rule": "exact_equation_prefix_plus_digit_presence",
                        "query_operator": query_op or "",
                        "example_operators": "".join(sorted(example_ops)),
                    }
                return {
                    "task_type": "equation_symbolic",
                    "equation_subtype": "symbolic",
                    "classification_confidence": "high",
                    "classification_rule": "exact_equation_prefix_plus_no_digits",
                    "query_operator": "",
                    "example_operators": "",
                }
            return {
                "task_type": base_label,
                "equation_subtype": "",
                "classification_confidence": "high",
                "classification_rule": rule,
                "query_operator": "",
                "example_operators": "",
            }
    return {
        "task_type": "unknown",
        "equation_subtype": "",
        "classification_confidence": "low",
        "classification_rule": "no_registered_template_prefix",
        "query_operator": "",
        "example_operators": "",
    }


classification_records = []
for row in raw_df.itertuples(index=False):
    result = classify_prompt(row.prompt)
    normalized = normalize_prompt(row.prompt)
    classification_records.append({
        "source_row": int(row.source_row),
        "source_dataset": row.source_dataset,
        "id": row.id,
        "original_prompt": row.prompt,
        "original_answer": row.answer,
        "prompt_hash": prompt_hash(row.prompt),
        "normalized_prompt_hash": prompt_hash(normalized),
        "has_digit": bool(re.search(r"\d", row.prompt)),
        "prompt_characters": int(len(row.prompt)),
        "answer_characters": int(len(row.answer)),
        **result,
        "kept_for_course": result["task_type"] in TARGET_TASKS,
        "exclusion_reason": "" if result["task_type"] in TARGET_TASKS else f"excluded_{result['task_type']}",
    })

audit_df = pd.DataFrame(classification_records)
print(audit_df["task_type"].value_counts(dropna=False).to_string())
print("\nEquation Numeric subtype:")
print(audit_df.loc[audit_df.task_type == "equation_numeric", "equation_subtype"].value_counts(dropna=False).to_string())


In [ ]:
print("各分类抽样（只显示 Prompt 首句和答案，不修改原始数据）：")
for task_type, group in raw_df.assign(task_type=audit_df["task_type"]).groupby("task_type", sort=True):
    print(f"\n### {task_type}: n={len(group)}")
    sample = group.sample(n=min(2, len(group)), random_state=SEED)
    for row in sample.itertuples(index=False):
        print({"id": row.id, "prompt_head": row.prompt.splitlines()[0], "answer": row.answer})

unknown_rows = audit_df[audit_df.task_type == "unknown"]
if len(unknown_rows):
    print("\nUNKNOWN 样本必须人工复核：")
    print(unknown_rows[["source_row", "id", "classification_rule"]].head(20).to_string(index=False))
else:
    print("\n没有未知模板；仍保留 unknown 分类路径以防数据版本改变。")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = audit_df["task_type"].value_counts().sort_values(ascending=True)
counts.plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Raw task distribution (exact template classification)")
ax.set_xlabel("Number of samples")
ax.set_ylabel("Task type")
for i, value in enumerate(counts):
    ax.text(value + max(counts) * 0.01, i, str(value), va="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "raw_task_distribution.png", dpi=150)
plt.show()


## 4. 清洗主线：只保留三个正式题型

清洗只保留 `gravity`、`equation_numeric` 和 `bit_manipulation`。`equation_symbolic`、Unit Conversion、Encryption、Numeral System 和 Unknown 都保留在 `classification_audit.csv` 中，并带有排除原因。

`real_cleaned.csv` 仍保留原始 ID、原始 Prompt、原始 Answer 和来源行号，后续训练数据可以从它追溯回官方文件。


In [ ]:
raw_with_classification = raw_df.join(audit_df.set_index("source_row"), on="source_row", rsuffix="_audit")
cleaned_df = raw_with_classification[raw_with_classification["task_type"].isin(sorted(TARGET_TASKS))].copy()
cleaned_columns = [
    "source_row", "source_dataset", "id", "prompt", "answer", "task_type", "equation_subtype",
    "query_operator", "example_operators", "prompt_hash", "normalized_prompt_hash",
    "classification_confidence", "classification_rule",
]
audit_path = OUTPUT_DIR / "classification_audit.csv"
cleaned_path = OUTPUT_DIR / "real_cleaned.csv"
audit_df.to_csv(audit_path, index=False)
cleaned_df[cleaned_columns].to_csv(cleaned_path, index=False)

cleaning_report = {
    "raw_rows": int(len(raw_df)),
    "kept_rows": int(len(cleaned_df)),
    "removed_rows": int(len(raw_df) - len(cleaned_df)),
    "kept_counts": {k: int(v) for k, v in cleaned_df.task_type.value_counts().to_dict().items()},
    "removed_counts": {k: int(v) for k, v in audit_df.loc[~audit_df.kept_for_course, "task_type"].value_counts().to_dict().items()},
    "audit_path": str(audit_path),
    "cleaned_path": str(cleaned_path),
}
print(json.dumps(cleaning_report, ensure_ascii=False, indent=2))


## 5. 划分与 Leakage Audit

默认划分是固定种子的**分层 IID Holdout**：每个正式题型分别按 80% / 10% / 10% 划分。它能提供同分布泛化观察，但不能自动证明不存在同源题目泄漏。

本数据没有公开的规则实例 ID、模板组 ID 或生成种子。我们会审计完全重复 Prompt 和把数字/二进制字面量替换后的规范化重复；如果规范化重复跨集合，只报告风险，不把它伪装成 Structural OOD 证据。Structural OOD 需要后续补充可靠的规则组合或结构家族字段后再做。


In [ ]:
def make_stratified_split(frame: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    assignments = {}
    for task_type, group in frame.groupby("task_type", sort=True):
        indices = group.index.to_numpy(copy=True)
        rng.shuffle(indices)
        n = len(indices)
        n_validation = max(1, int(round(n * 0.10)))
        n_test = max(1, int(round(n * 0.10)))
        for index in indices[: n - n_validation - n_test]:
            assignments[index] = "train"
        for index in indices[n - n_validation - n_test : n - n_test]:
            assignments[index] = "validation"
        for index in indices[n - n_test :]:
            assignments[index] = "holdout_test"
    result = frame.copy()
    result["split"] = pd.Series(assignments)
    if result["split"].isna().any():
        raise AssertionError("存在未分配 split 的样本")
    return result

split_df = make_stratified_split(cleaned_df, seed=SEED)
for split_name, split_group in split_df.groupby("split", sort=True):
    split_group[cleaned_columns + ["split"]].to_csv(SPLIT_DIR / f"{split_name}.csv", index=False)


def cross_split_duplicate_counts(frame: pd.DataFrame, key: str) -> dict[str, object]:
    owners = frame.groupby(key)["split"].agg(lambda values: sorted(set(values)))
    leaked = owners[owners.map(len) > 1]
    return {
        "key": key,
        "duplicate_keys": int((owners.map(len) > 1).sum()),
        "rows_in_cross_split_keys": int(frame[frame[key].isin(leaked.index)].shape[0]),
        "examples": [{"key": str(k), "splits": v} for k, v in leaked.head(5).items()],
    }

split_counts = (
    split_df.groupby(["split", "task_type"], sort=True).size().unstack(fill_value=0).astype(int)
)
leakage_audit = {
    "exact_id": cross_split_duplicate_counts(split_df, "id"),
    "exact_prompt": cross_split_duplicate_counts(split_df, "prompt_hash"),
    "normalized_prompt": cross_split_duplicate_counts(split_df, "normalized_prompt_hash"),
    "group_aware_split": {
        "status": "not_available",
        "reason": "原始 CSV 未提供可靠的规则实例、生成种子或结构家族字段；规范化 Prompt 只能用于审计，不能作为同源组的充分证明。",
    },
}
print(split_counts.to_string())
print(json.dumps(leakage_audit, ensure_ascii=False, indent=2))


In [ ]:
def prompt_token_proxy(text: str) -> int:
    # 这是不依赖模型 Tokenizer 的审计量，不应冒充 Qwen3 token 数。
    return len(re.findall(r"\w+|[^\w\s]", str(text), flags=re.UNICODE))

split_df["prompt_token_proxy"] = split_df["prompt"].map(prompt_token_proxy)
rule_distribution = {
    "task_type": {k: int(v) for k, v in split_df.task_type.value_counts().to_dict().items()},
    "equation_subtype": {k: int(v) for k, v in split_df.loc[split_df.task_type == "equation_numeric", "equation_subtype"].value_counts().to_dict().items()},
    "equation_query_operator": {k: int(v) for k, v in split_df.loc[split_df.task_type == "equation_numeric", "query_operator"].value_counts().to_dict().items()},
}

dataset_statistics = {
    "schema_version": "nemotron-course-dataset/v1",
    "source": {"path": str(RAW_PATH.relative_to(PROJECT_ROOT)), "dataset": SOURCE_DATASET, "sha256": raw_profile["sha256"]},
    "seed": SEED,
    "raw_profile": raw_profile,
    "classification_counts": {k: int(v) for k, v in audit_df.task_type.value_counts().to_dict().items()},
    "cleaned_counts": {k: int(v) for k, v in cleaned_df.task_type.value_counts().to_dict().items()},
    "split_counts": {
        split: {task: int(value) for task, value in row.items()}
        for split, row in split_counts.iterrows()
    },
    "rule_distribution": rule_distribution,
    "length_summary": {
        split: {
            "rows": int(len(group)),
            "prompt_char_mean": float(group.prompt.str.len().mean()),
            "prompt_token_proxy_mean": float(group.prompt_token_proxy.mean()),
            "answer_char_mean": float(group.answer.str.len().mean()),
        }
        for split, group in split_df.groupby("split", sort=True)
    },
    "leakage_audit": leakage_audit,
    "limitations": [
        "prompt_token_proxy 不是 Qwen3 Tokenizer 的真实 token 数；真实 token 统计需在官方训练环境加载 tokenizer 后补充。",
        "没有可靠的规则实例/结构家族键，因此当前只提供 IID Holdout，不宣称 Structural OOD 已完成。",
        "规范化 Prompt 重复是风险信号，不等同于同源题目证明。",
    ],
}
(OUTPUT_DIR / "dataset_statistics.json").write_text(json.dumps(dataset_statistics, ensure_ascii=False, indent=2), encoding="utf-8")
split_manifest = {
    "schema_version": "nemotron-course-split/v1",
    "seed": SEED,
    "ratios_requested": {"train": 0.8, "validation": 0.1, "holdout_test": 0.1},
    "method": "per-task stratified IID split",
    "structural_ood": "not_available_without_reliable_group_key",
    "files": {name: str((SPLIT_DIR / f"{name}.csv").relative_to(PROJECT_ROOT)) for name in ["train", "validation", "holdout_test"]},
    "leakage_audit": leakage_audit,
}
(OUTPUT_DIR / "split_manifest.json").write_text(json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({"statistics": str(OUTPUT_DIR / 'dataset_statistics.json'), "split_manifest": str(OUTPUT_DIR / 'split_manifest.json')}, ensure_ascii=False, indent=2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_counts = split_df.groupby(["split", "task_type"]).size().unstack(fill_value=0)
plot_counts.plot.bar(ax=ax, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Stratified IID split by task type")
ax.set_xlabel("Split")
ax.set_ylabel("Number of samples")
ax.legend(title="Task type")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "split_task_distribution.png", dpi=150)
plt.show()


## 6. Baseline：原始 Prompt、统一生成和确定性验证

Baseline 必须回答“未经 Prompt Engineering、SFT 或 RL 的 Qwen3 基座模型表现如何”。因此：

- 输入只使用原始比赛 Prompt；不注入课程先验规则库。
- 生成使用 Qwen3 官方 Chat Template，并在模型支持时显式开启 Thinking。
- 评测输出保留原文、提取答案、提取状态、生成 token 数和耗时。
- 不把没有运行的分数写成结果。

答案提取器只负责把模型回答映射到候选答案；Verifier 再按题型比较。对无法提取或格式不合法的输出，计入 Invalid Output，而不是当作错误答案静默吞掉。


In [ ]:
def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=re.I)
    return text.strip()


def extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?:final|answer)>\s*(.*?)\s*</(?:final|answer)>", text, flags=re.I | re.S)
    if tagged:
        return normalize_answer_text(tagged[-1]), "tagged"
    if task_type == "bit_manipulation":
        matches = re.findall(r"(?<![01])[01]{8}(?![01])", text)
        if matches:
            return matches[-1], "binary_fallback"
    if task_type in {"gravity", "equation_numeric"}:
        matches = re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", text)
        if matches:
            return matches[-1], "numeric_fallback"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        return normalize_answer_text(lines[-1]), "last_nonempty_line"
    return None, "empty"


OFFICIAL_VERIFIER_VERSION = "official_nvidia_metric_v2"
OFFICIAL_NUMERIC_REL_TOL = 1e-2
OFFICIAL_NUMERIC_ABS_TOL = 1e-5


def official_numeric_match(predicted: object, expected: object) -> bool:
    """NVIDIA metric-compatible numeric comparison for answers and Verifiers."""
    try:
        predicted_value = float(normalize_answer_text(predicted))
        expected_value = float(normalize_answer_text(expected))
    except (TypeError, ValueError):
        return False
    if not (math.isfinite(predicted_value) and math.isfinite(expected_value)):
        return False
    return math.isclose(
        predicted_value,
        expected_value,
        rel_tol=OFFICIAL_NUMERIC_REL_TOL,
        abs_tol=OFFICIAL_NUMERIC_ABS_TOL,
    )


def numeric_or_text_match(predicted: object, expected: object) -> bool:
    """Use the official tolerance for numeric strings and exact matching otherwise."""
    predicted_text, expected_text = normalize_answer_text(predicted), normalize_answer_text(expected)
    if official_numeric_match(predicted_text, expected_text):
        return True
    return predicted_text == expected_text


def verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text = normalize_answer_text(predicted)
    expected_text = normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p = predicted_text.removeprefix("0b")
        e = expected_text.removeprefix("0b")
        if not re.fullmatch(r"[01]{8}", p) or not re.fullmatch(r"[01]{8}", e):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", predicted_text):
            return False, "invalid_numeric_format"
        return official_numeric_match(predicted_text, expected_text), "official_math_isclose"
    return predicted_text == expected_text, "exact_text"


def summarize_baseline_records(records: list[dict[str, object]]) -> dict[str, object]:
    frame = pd.DataFrame(records)
    if frame.empty:
        return {"records": 0, "status": "no_records"}
    summaries = {}
    for task_type, group in frame.groupby("task_type", sort=True):
        summaries[task_type] = {
            "records": int(len(group)),
            "accuracy": float(group["correct"].mean()),
            "answer_extraction_success_rate": float(group["extraction_success"].mean()),
            "invalid_output_rate": float((~group["extraction_success"]).mean()),
            "average_generation_tokens": float(group["generation_tokens"].mean()),
            "average_inference_time_seconds": float(group["inference_time_seconds"].mean()),
            "total_inference_time_seconds": float(group["inference_time_seconds"].sum()),
        }
    task_accuracy = frame.groupby("task_type")["correct"].mean()
    return {
        "records": int(len(frame)),
        "overall_accuracy_sample_weighted": float(frame["correct"].mean()),
        "macro_accuracy": float(task_accuracy.mean()),
        "by_task": summaries,
    }


In [ ]:
def render_qwen3_prompt(tokenizer, original_prompt: str) -> str:
    messages = [{"role": "user", "content": original_prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
    except TypeError:
        # 兼容旧版 Transformers：仍使用官方模板，但不传新参数。
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def run_vllm_baseline(
    evaluation_frame: pd.DataFrame,
    model_id: str = MODEL_ID,
    max_new_tokens: int = MAX_NEW_TOKENS,
    output_path: Path = OUTPUT_DIR / "baseline_results.jsonl",
) -> dict[str, object]:
    """Run the original-prompt Baseline with the official vLLM profile."""
    try:
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams
        import torch
        import subprocess
        import threading
    except ImportError as exc:
        return {"status": "blocked_missing_vllm_profile", "reason": repr(exc)}

    model_source = os.environ.get("LOCAL_MODEL_PATH", model_id)
    local_files_only = Path(model_source).exists()
    tokenizer = AutoTokenizer.from_pretrained(model_source, local_files_only=local_files_only, trust_remote_code=True)
    prompts = [render_qwen3_prompt(tokenizer, p) for p in evaluation_frame["prompt"]]
    max_model_len = VLLM_MAX_MODEL_LEN
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # vLLM keeps most allocations outside torch.cuda memory accounting. Sample
    # nvidia-smi in a background thread so the report contains the device peak;
    # the initial value and delta make concurrent workloads visible.
    gpu_memory_samples: list[float] = []
    monitor_stop = threading.Event()

    def _sample_nvidia_smi_memory() -> None:
        while not monitor_stop.is_set():
            try:
                raw = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                    text=True,
                    stderr=subprocess.DEVNULL,
                    timeout=2,
                )
                values = [float(line.strip()) for line in raw.splitlines() if line.strip()]
                if values:
                    gpu_memory_samples.append(float(sum(values)))
            except (OSError, subprocess.SubprocessError, ValueError):
                pass
            monitor_stop.wait(0.25)

    monitor_thread = threading.Thread(target=_sample_nvidia_smi_memory, daemon=True)
    monitor_thread.start()
    load_started = time.perf_counter()
    llm = LLM(
        model=model_source,
        tokenizer=model_source,
        tensor_parallel_size=1,
        gpu_memory_utilization=float(os.environ.get("VLLM_GPU_MEMORY_UTILIZATION", "0.85")),
        max_model_len=max_model_len,
        enforce_eager=True,
    )
    load_seconds = time.perf_counter() - load_started
    sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_new_tokens)
    started = time.perf_counter()
    outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    elapsed_total = time.perf_counter() - started
    monitor_stop.set()
    monitor_thread.join(timeout=2)
    peak_memory_mib = float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else None
    initial_gpu_memory_mib = float(gpu_memory_samples[0]) if gpu_memory_samples else None
    peak_gpu_memory_mib = float(max(gpu_memory_samples)) if gpu_memory_samples else None
    peak_gpu_memory_delta_mib = (
        float(peak_gpu_memory_mib - initial_gpu_memory_mib)
        if peak_gpu_memory_mib is not None and initial_gpu_memory_mib is not None else None
    )

    records = []
    for row, request_output in zip(evaluation_frame.itertuples(index=False), outputs):
        generated = request_output.outputs[0]
        text = generated.text
        extracted, extraction_method = extract_final_answer(text, row.task_type)
        correct, verification_method = verify_answer(row.task_type, extracted, row.answer)
        token_ids = getattr(generated, "token_ids", ()) or ()
        finish_reason = getattr(generated, "finish_reason", None)
        prompt_token_ids = getattr(request_output, "prompt_token_ids", ()) or ()
        records.append({
            "id": row.id,
            "source_row": int(row.source_row),
            "task_type": row.task_type,
            "prompt": row.prompt,
            "expected_answer": row.answer,
            "raw_output": text,
            "extracted_answer": extracted,
            "extraction_method": extraction_method,
            "verification_method": verification_method,
            "verifier_version": OFFICIAL_VERIFIER_VERSION,
            "extraction_success": extracted is not None,
            "correct": bool(correct),
            "prompt_tokens": int(len(prompt_token_ids)),
            "generation_tokens": int(len(token_ids)),
            "finish_reason": finish_reason,
            "truncated": bool(finish_reason == "length" or len(token_ids) >= max_new_tokens),
            "inference_time_seconds": float(elapsed_total / max(len(outputs), 1)),
        })
    payload = "\n".join(json.dumps(r, ensure_ascii=False) for r in records)
    output_path.write_text(payload + ("\n" if payload else ""), encoding="utf-8")
    summary = summarize_baseline_records(records)
    total_output_tokens = int(sum(record["generation_tokens"] for record in records))
    summary.update({
        "model_id": model_id,
        "model_source": model_source,
        "max_model_len": max_model_len,
        "max_new_tokens": max_new_tokens,
        "load_time_seconds": float(load_seconds),
        "total_generation_time_seconds": float(elapsed_total),
        "total_output_tokens": total_output_tokens,
        "throughput_output_tokens_per_second": float(total_output_tokens / elapsed_total) if elapsed_total else 0.0,
        "truncation_rate": float(np.mean([record["truncated"] for record in records])) if records else 0.0,
        "peak_memory_allocated_mib": peak_memory_mib,
        "initial_gpu_memory_used_mib": initial_gpu_memory_mib,
        "peak_gpu_memory_used_mib": peak_gpu_memory_mib,
        "peak_gpu_memory_delta_mib": peak_gpu_memory_delta_mib,
        "verification_standard": {"numeric": "math.isclose(rel_tol=1e-2, abs_tol=1e-5)", "bit": "exact_8bit"},
    })
    (OUTPUT_DIR / "baseline_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    del llm
    try:
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return {"status": "completed", "summary": summary, "results_path": str(output_path)}


if RUN_BASELINE:
    baseline_frame = split_df[split_df["split"] == ("validation" if RUN_MODE == "smoke" else "holdout_test")].copy()
    if RUN_MODE == "smoke":
        baseline_frame = pd.concat([
            group.head(MAX_SAMPLES_PER_TASK) for _, group in baseline_frame.groupby("task_type", sort=True)
        ], ignore_index=True)
    baseline_status = run_vllm_baseline(baseline_frame)
else:
    baseline_status = {
        "status": "skipped_by_default",
        "reason": "RUN_BASELINE=False；未下载模型、未占用 GPU、未生成虚构分数。",
        "next_action": "在官方 vLLM Profile 和 RTX 5090 可用时改为 True，并先运行 Validation smoke。",
    }
print(json.dumps(baseline_status, ensure_ascii=False, indent=2))


## 7. 可选数据工程：Prior → Generator → Solver → Verifier → Screening

真实数据清洗与划分已经在前面的主线完成。这里使用固定种子从 Train 侧先验生成三类 Pilot，每类 100 个**单次生成候选**；每次尝试都保留 `candidate_id`、`generation_status`、`generation_error`、Solver 输出和拒绝原因。合成过程不读取 Validation / Holdout Test 的答案。

- **Gravity**：先对 Train 的 1,277 道题做 `g_i`、OLS、Median、时间、示例数、残差和小数位审计；新 Generator 从 Train 经验分布采样，不再使用固定 9.50–19.50 均匀网格。Verifier 使用独立 OLS 浮点路径和官方 `math.isclose` 标准。
- **Equation Numeric**：采用参考实现中的 24 条 `(parse, operation, format)` 规则，Operator 池从 Train 实际提取的 26 个符号构成。一个 Prompt 同时包含多个运算符，保留 normal / flip family、semantic group 和 Join 全局约束。Guess 分为 `guess_rule_unique`、`guess_answer_equivalent` 和 `guess_answer_ambiguous`；第三类拒绝进入训练集。
- **Bit Manipulation**：使用课程规定的 8-bit DSL。Prompt 只呈现输入输出示例和 Query；Solver 与 Verifier 使用独立表达式解析/计算路径，枚举深度不超过 2 的表达式并按完整 256 输入语义去重，分别记录表达式唯一性和答案唯一性。

官方数值答案和合成数值验证统一使用 `math.isclose(predicted, expected, rel_tol=1e-2, abs_tol=1e-5)`；Bit Manipulation 仍精确匹配 8-bit 字符串。每条候选都保留 hidden rule、Solver answer、trace 和 verification status，这些元数据不进入模型 Prompt。另有负样本测试覆盖错误答案、错误轨迹、非法表达式、不可识别问题和重复 Prompt。

### 可选 NeMo Data Designer 分支

根据 NVIDIA-NeMo/DataDesigner 当前官方 API，使用 `ModelProvider(provider_type="openai")` 指向本地 vLLM `http://127.0.0.1:8000/v1`，`ModelConfig` 使用 `Qwen3-4B-Thinking-2507`，不使用 `NVIDIA_API_KEY`、托管模型或外部 API。Preview 采用 Gravity / Equation Numeric / Bit Manipulation 各 3 条的均衡 Seed，Seed 只含 `task_type` 和 `prompt`。Data Designer 只扩充语言表达和候选轨迹，全部输出仍需经过确定性 Solver / Verifier，且只能保留为 Preview 产物。


In [ ]:
# 合成分支的配置只使用课程先验和固定随机种子；不读取 Validation / Holdout Test。
from itertools import product
from functools import lru_cache
from datetime import datetime, timezone
import os

SYNTHETIC_DIR = OUTPUT_DIR / "synthetic"
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
PILOT_CANDIDATES_PER_TASK = SYNTHETIC_SAMPLES_PER_TASK
RUN_SYNTHETIC = os.environ.get("RUN_SYNTHETIC", "0") == "1"
RUN_DATA_DESIGNER_PREVIEW = os.environ.get("RUN_DATA_DESIGNER_PREVIEW", "0") == "1"
GENERATOR_VERSION = "nemotron-synthetic-v3"

GRAVITY_PRIOR = {
    "formula": "distance = 0.5 * g * t^2",
    "units": {"t": "seconds", "distance": "meters", "g": "m/s^2"},
    "sampling_source": "train_split_empirical_profile",
    "answer_precision": "observed_decimal_places_with_official_tolerance",
}
BIT_PRIOR = {
    "width": 8,
    "operations": ["SHL", "SHR", "ROTL", "ROTR", "AND", "OR", "XOR", "NOT"],
    "expression_language": "x is the input byte; unary and binary trees have maximum depth 2",
    "mask": 255,
}

# Numeric Equation prior extracted from the reference solver's DSL and Numeric24 rule table.
# A rule is (operand parse, arithmetic operation, output format); it is not merely an
# operator -> answer lookup. Global assignments additionally enforce one family and one
# non-join semantic group per prompt, while join rules may coexist.
NUMERIC24_RULES = [
    {"name": "sub", "parse": "AB_CD", "operation": "sub_lr", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff", "parse": "AB_CD", "operation": "abs_sub", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff_prefix", "parse": "AB_CD", "operation": "abs_sub", "format": "sign_prefix_raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "join", "parse": "AB_CD", "operation": "concat_lr", "format": "raw", "family": "join", "group": "join_like"},
    {"name": "join_swap", "parse": "AB_CD", "operation": "concat_rl", "format": "raw", "family": "join", "group": "join_like"},
    {"name": "add", "parse": "AB_CD", "operation": "add", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "add_p1", "parse": "AB_CD", "operation": "add_p1", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "add_m1", "parse": "AB_CD", "operation": "add_m1", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "mul", "parse": "AB_CD", "operation": "mul", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "mul_p1", "parse": "AB_CD", "operation": "mul_p1", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "mul_m1", "parse": "AB_CD", "operation": "mul_m1", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "maxmodmin", "parse": "AB_CD", "operation": "max_mod_min", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "flip_add", "parse": "BA_DC", "operation": "add", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_add_p1", "parse": "BA_DC", "operation": "add_p1", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_add_m1", "parse": "BA_DC", "operation": "add_m1", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_mul", "parse": "BA_DC", "operation": "mul", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_p1", "parse": "BA_DC", "operation": "mul_p1", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_m1", "parse": "BA_DC", "operation": "mul_m1", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_absdiff", "parse": "BA_DC", "operation": "abs_sub", "format": "rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_maxmodmin", "parse": "BA_DC", "operation": "max_mod_min", "format": "rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_prefix", "parse": "BA_DC", "operation": "sub_lr", "format": "sign_prefix_rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_suffix", "parse": "BA_DC", "operation": "sub_rl", "format": "rev_or_neg_suffix", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_prefix", "parse": "BA_DC", "operation": "abs_sub", "format": "always_prefix_rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_suffix", "parse": "BA_DC", "operation": "abs_sub", "format": "always_suffix_rev", "family": "flip", "group": "sub_or_mod_like"},
]
NUMERIC24_RULE_BY_NAME = {rule["name"]: rule for rule in NUMERIC24_RULES}
NUMERIC24_FORMULAS = {
    "sub": "AB - CD, raw", "absdiff": "|AB - CD|, raw", "absdiff_prefix": "|AB - CD| with operator prefix",
    "join": "AB concatenated with CD", "join_swap": "CD concatenated with AB",
    "add": "AB + CD", "add_p1": "AB + CD + 1", "add_m1": "AB + CD - 1",
    "mul": "AB * CD", "mul_p1": "AB * CD + 1", "mul_m1": "AB * CD - 1",
    "maxmodmin": "max(AB, CD) mod min(AB, CD)",
    "flip_add": "reverse(BA + DC)", "flip_add_p1": "reverse(BA + DC + 1)", "flip_add_m1": "reverse(BA + DC - 1)",
    "flip_mul": "reverse(BA * DC)", "flip_mul_p1": "reverse(BA * DC + 1)", "flip_mul_m1": "reverse(BA * DC - 1)",
    "flip_absdiff": "reverse(|BA - DC|)", "flip_maxmodmin": "reverse(max(BA, DC) mod min(BA, DC))",
    "flip_sub_prefix": "reverse(BA - DC), preserving a negative sign as an operator prefix",
    "flip_sub_suffix": "reverse(DC - BA), appending the operator for a negative result",
    "flip_absdiff_prefix": "operator-prefixed reverse(|BA - DC|)",
    "flip_absdiff_suffix": "reverse(|BA - DC|) followed by the operator",
}
EQUATION_NUMERIC_PRIOR = {
    "operand_pattern": "AB<operator>CD with two decimal digits on each side",
    "parses": {"AB_CD": "left=AB, right=CD", "BA_DC": "left=reverse(AB), right=reverse(CD)"},
    "operations": ["concatenate", "add", "add±1", "subtract", "reverse subtract", "absolute difference", "multiply", "multiply±1", "max mod min"],
    "output_formats": ["raw", "reverse digits", "operator-prefixed reverse", "two-digit padding", "operator suffix"],
    "rules": [{"name": name, "formula": NUMERIC24_FORMULAS[name]} for name in NUMERIC24_RULE_BY_NAME],
    "global_constraints": [
        "All non-join rules in one prompt use either normal or flipped operand family.",
        "At most one non-join rule from each semantic group: sub_or_mod_like, add_like, mul_like.",
        "Join and reverse-join are the join_like exception and can coexist with the other groups.",
        "A query operator absent from examples is Guess; it is admitted only when all valid assignments give one query output.",
    ],
}


# The synthetic distribution is fitted from the Train split only. Validation and
# Holdout rows never enter these arrays or the generator seed.
GRAVITY_PAIR_RE = re.compile(r"For t = ([-+]?\d+(?:\.\d+)?)s, distance = ([-+]?\d+(?:\.\d+)?) m")


def _gravity_quantiles(values: list[float]) -> dict[str, float]:
    values = [float(value) for value in values]
    quantile_levels = {"q01": 0.01, "q05": 0.05, "q25": 0.25, "median": 0.50, "q75": 0.75, "q95": 0.95, "q99": 0.99}
    array = np.asarray(values, dtype=float)
    return {
        "count": int(array.size),
        "min": float(array.min()),
        **{name: float(np.quantile(array, level)) for name, level in quantile_levels.items()},
        "max": float(array.max()),
        "mean": float(array.mean()),
        "std": float(array.std()),
    }


def _extract_gravity_profile(frame: pd.DataFrame) -> dict[str, object]:
    records = []
    for row in frame.itertuples(index=False):
        pairs = [(float(t), float(distance)) for t, distance in GRAVITY_PAIR_RE.findall(str(row.prompt))]
        if len(pairs) < 2:
            continue
        times = np.asarray([t for t, _ in pairs], dtype=float)
        distances = np.asarray([distance for _, distance in pairs], dtype=float)
        g_i = 2.0 * distances / np.square(times)
        g_ols = 2.0 * float(np.sum(np.square(times) * distances)) / float(np.sum(np.power(times, 4)))
        g_median = float(np.median(g_i))
        residuals = distances - 0.5 * g_ols * np.square(times)
        decimals = [len(str(distance).split(".", 1)[1]) if "." in str(distance) else 0 for _, distance in pairs]
        records.append({
            "source_row": int(getattr(row, "source_row", -1)),
            "id": str(getattr(row, "id", "")),
            "example_count": len(pairs),
            "times": times.tolist(),
            "distances": distances.tolist(),
            "g_i": g_i.tolist(),
            "g_ols": g_ols,
            "g_median": g_median,
            "residuals": residuals.tolist(),
            "distance_decimal_places": decimals,
        })
    if not records:
        raise ValueError("No parseable Gravity rows in Train split")
    return {
        "source_split": "train",
        "row_count": len(records),
        "g_ols_values": [record["g_ols"] for record in records],
        "g_median_values": [record["g_median"] for record in records],
        "t_values": [t for record in records for t in record["times"]],
        "d_values": [d for record in records for d in record["distances"]],
        "example_counts": [record["example_count"] for record in records],
        "residual_values": [residual for record in records for residual in record["residuals"]],
        "distance_decimal_places": [places for record in records for places in record["distance_decimal_places"]],
        "audit": {
            "example_count_distribution": {str(k): int(v) for k, v in Counter(record["example_count"] for record in records).items()},
            "t": _gravity_quantiles([t for record in records for t in record["times"]]),
            "distance": _gravity_quantiles([d for record in records for d in record["distances"]]),
            "g_i": _gravity_quantiles([g for record in records for g in record["g_i"]]),
            "g_ols": _gravity_quantiles([record["g_ols"] for record in records]),
            "g_median": _gravity_quantiles([record["g_median"] for record in records]),
            "residual": _gravity_quantiles([residual for record in records for residual in record["residuals"]]),
            "distance_decimal_places": {str(k): int(v) for k, v in Counter(place for record in records for place in record["distance_decimal_places"]).items()},
        },
    }


train_gravity_frame = split_df[(split_df["split"] == "train") & (split_df["task_type"] == "gravity")].copy()
GRAVITY_TRAIN_PROFILE = _extract_gravity_profile(train_gravity_frame)
GRAVITY_PRIOR["observed_train_profile"] = GRAVITY_TRAIN_PROFILE["audit"]
GRAVITY_AUDIT_PATH = OUTPUT_DIR / "gravity_distribution_audit.json"
GRAVITY_AUDIT_PATH.write_text(json.dumps({
    "schema_version": "nemotron-course-gravity-audit/v1",
    "source_split": "train",
    "source_rows": GRAVITY_TRAIN_PROFILE["row_count"],
    "formula": "g_i=2*d_i/t_i^2; g_ols=2*sum(t_i^2*d_i)/sum(t_i^4)",
    "statistics": GRAVITY_TRAIN_PROFILE["audit"],
    "generator_sampling": {
        "g": "empirical Train g_ols values",
        "t": "empirical Train example times",
        "example_count": "empirical Train per-question counts",
        "distance_precision": "empirical Train decimal-place frequencies",
        "noise": "no additional noise; distance is rounded at sampled observed precision",
    },
}, ensure_ascii=False, indent=2), encoding="utf-8")

legacy_g_grid = np.arange(9.50, 19.50 + 0.005, 0.01)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(GRAVITY_TRAIN_PROFILE["g_ols_values"], bins=30, alpha=0.75, label="Train g OLS", color="#4C78A8")
axes[0, 0].hist(legacy_g_grid, bins=30, alpha=0.35, label="legacy uniform grid", color="#F58518")
axes[0, 0].set_title("Gravity g distribution: Train vs legacy")
axes[0, 0].set_xlabel("estimated g")
axes[0, 0].set_ylabel("count")
axes[0, 0].legend()
axes[0, 1].hist(GRAVITY_TRAIN_PROFILE["t_values"], bins=30, color="#54A24B")
axes[0, 1].set_title("Train example time distribution")
axes[0, 1].set_xlabel("t (s)")
axes[0, 1].set_ylabel("count")
axes[1, 0].hist(GRAVITY_TRAIN_PROFILE["d_values"], bins=30, color="#B279A2")
axes[1, 0].set_title("Train example distance distribution")
axes[1, 0].set_xlabel("distance (m)")
axes[1, 0].set_ylabel("count")
axes[1, 1].hist(GRAVITY_TRAIN_PROFILE["residual_values"], bins=30, color="#E45756")
axes[1, 1].set_title("OLS fit residual distribution")
axes[1, 1].set_xlabel("distance residual (m)")
axes[1, 1].set_ylabel("count")
fig.tight_layout()
GRAVITY_COMPARISON_PLOT = OUTPUT_DIR / "gravity_distribution_comparison.png"
fig.savefig(GRAVITY_COMPARISON_PLOT, dpi=150)
plt.show()


def _extract_train_numeric_operators(frame: pd.DataFrame) -> tuple[str, ...]:
    operators = set()
    for prompt in frame.loc[frame["task_type"] == "equation_numeric", "prompt"]:
        for line in str(prompt).splitlines():
            example = re.match(r"^\s*\d{2}([^\d\s])\d{2}\s*=", line)
            query = re.search(r"Now, determine the result for:\s*\d{2}([^\d\s])\d{2}", line)
            if example:
                operators.add(example.group(1))
            if query:
                operators.add(query.group(1))
    return tuple(sorted(operators))


TRAIN_NUMERIC_OPERATOR_POOL = _extract_train_numeric_operators(split_df[split_df["split"] == "train"])
NUMERIC_OPERATOR_INVENTORY = {
    "train_observed": list(TRAIN_NUMERIC_OPERATOR_POOL),
    "count": len(TRAIN_NUMERIC_OPERATOR_POOL),
    "unsupported_by_prior": [],
}
(OUTPUT_DIR / "equation_operator_inventory.json").write_text(json.dumps(NUMERIC_OPERATOR_INVENTORY, ensure_ascii=False, indent=2), encoding="utf-8")


print(json.dumps({"gravity_prior": GRAVITY_PRIOR, "bit_prior": BIT_PRIOR, "equation_numeric_rules": len(NUMERIC24_RULES), "pilot_candidates_per_task": PILOT_CANDIDATES_PER_TASK}, ensure_ascii=False, indent=2))


In [ ]:
# Gravity: one-shot candidate generator, prompt solver, and independent verifier.
# The verifier has its own floating-point fit and never calls generator helpers.
def _gravity_generator_render(distance_t: float | Decimal, g_value: float | Decimal, decimal_places: int = 2) -> str:
    t = Decimal(str(distance_t))
    g = Decimal(str(g_value))
    quantizer = Decimal(1).scaleb(-int(decimal_places))
    return format((Decimal("0.5") * g * t * t).quantize(quantizer), "f")


# Kept as a named generator-side helper for traceability; verifier code below does not call it.
def _gravity_render(distance_t: float | Decimal, g_value: float | Decimal) -> str:
    return _gravity_generator_render(distance_t, g_value, decimal_places=2)


def _gravity_prompt_parts(prompt: str) -> tuple[list[tuple[str, str]], str | None]:
    examples = [(t, d) for t, d in re.findall(r"For t = ([-+]?\d+(?:\.\d+)?)s, distance = ([-+]?\d+(?:\.\d+)?) m", str(prompt))]
    query_match = re.search(r"falling distance for t = ([-+]?\d+(?:\.\d+)?)s", str(prompt))
    return examples, query_match.group(1) if query_match else None


def solve_gravity_prompt(prompt: str) -> dict[str, object]:
    """Solver: infer g from the public examples using OLS, then answer the query."""
    examples, query_t = _gravity_prompt_parts(prompt)
    if len(examples) < 2 or query_t is None:
        return {"status": "invalid_prompt", "answer": None, "inferred_g": None, "trace": {}}
    times = np.asarray([float(t) for t, _ in examples], dtype=float)
    distances = np.asarray([float(d) for _, d in examples], dtype=float)
    denominator = float(np.sum(np.power(times, 4)))
    if denominator <= 0:
        return {"status": "invalid_prompt", "answer": None, "inferred_g": None, "trace": {}}
    g_ols = 2.0 * float(np.sum(np.square(times) * distances)) / denominator
    query_raw = 0.5 * g_ols * float(query_t) ** 2
    answer = f"{query_raw:.2f}"
    trace = {
        "solver": "ols_from_public_examples",
        "g_estimates": [2.0 * d / (t * t) for t, d in zip(times.tolist(), distances.tolist())],
        "g_ols": g_ols,
        "query_t": str(query_t),
        "query_raw": query_raw,
    }
    return {"status": "solved", "answer": answer, "inferred_g": g_ols, "trace": trace}


def _gravity_verifier_fit(examples: list[tuple[str, str]]) -> dict[str, object]:
    """Independent verifier-side parameter fit; no generator or Solver call."""
    try:
        times = np.asarray([float(t) for t, _ in examples], dtype=float)
        distances = np.asarray([float(d) for _, d in examples], dtype=float)
    except (TypeError, ValueError):
        return {"valid": False, "g_ols": None, "g_median": None, "residuals": []}
    if len(times) < 2 or np.any(times <= 0) or not np.all(np.isfinite(distances)):
        return {"valid": False, "g_ols": None, "g_median": None, "residuals": []}
    denominator = float(np.sum(np.power(times, 4)))
    if denominator <= 0:
        return {"valid": False, "g_ols": None, "g_median": None, "residuals": []}
    per_example = 2.0 * distances / np.square(times)
    g_ols = 2.0 * float(np.sum(np.square(times) * distances)) / denominator
    g_median = float(np.median(per_example))
    residuals = (distances - 0.5 * g_ols * np.square(times)).tolist()
    return {
        "valid": True,
        "g_ols": g_ols,
        "g_median": g_median,
        "g_estimates": per_example.tolist(),
        "residuals": residuals,
        "times": times.tolist(),
        "distances": distances.tolist(),
    }


def _gravity_verifier_distance(time_value: float, g_value: float) -> float:
    # Separate implementation from Decimal-based generator rendering.
    return 0.5 * float(g_value) * float(time_value) * float(time_value)


def gravity_identifiability(prompt: str) -> dict[str, object]:
    examples, query_t = _gravity_prompt_parts(prompt)
    fit = _gravity_verifier_fit(examples)
    if not fit["valid"] or query_t is None:
        return {"candidate_g_count": 0, "query_output_count": 0, "query_outputs": [], "answer_unique": False}
    candidates = [float(fit["g_ols"]), float(fit["g_median"])]
    query_outputs = [_gravity_verifier_distance(float(query_t), g) for g in candidates]
    unique = all(official_numeric_match(query_outputs[0], value) for value in query_outputs[1:])
    return {
        "candidate_g_count": len(candidates),
        "candidate_g_values": candidates,
        "query_output_count": 1 if unique else len(query_outputs),
        "query_outputs": [query_outputs[0]] if unique else query_outputs,
        "answer_unique": unique,
    }


def generate_gravity_sample(rng: np.random.Generator) -> dict[str, object]:
    """Generator: one attempt from the empirical Train-only Gravity profile."""
    profile = GRAVITY_TRAIN_PROFILE
    example_count = int(rng.choice(np.asarray(profile["example_counts"], dtype=int)))
    g_value = float(rng.choice(np.asarray(profile["g_ols_values"], dtype=float)))
    times = rng.choice(np.asarray(profile["t_values"], dtype=float), size=example_count + 1, replace=False)
    decimal_places = rng.choice(np.asarray(profile["distance_decimal_places"], dtype=int), size=example_count, replace=True)
    example_pairs = [
        (f"{float(t):g}", _gravity_generator_render(float(t), g_value, int(places)))
        for t, places in zip(times[:-1], decimal_places)
    ]
    query_t = f"{float(times[-1]):g}"
    prompt = (
        "The course prior defines the physical relation distance = 0.5*g*t^2, with t in seconds and distance in meters.\n"
        + "\n".join(f"For t = {t}s, distance = {distance} m" for t, distance in example_pairs)
        + f"\nNow, calculate the falling distance for t = {query_t}s."
    )
    return {
        "task_type": "gravity",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"g": repr(g_value), "formula": GRAVITY_PRIOR["formula"], "profile_source": "train"},
        "prompt": prompt,
        "answer": "",
        "trace": {},
        "difficulty": f"examples_{example_count}",
        "deduce_or_guess": "deduce",
        "generation_status": "generated",
        "generation_error": None,
    }


def verify_gravity_sample(sample: dict[str, object]) -> dict[str, object]:
    """Verifier: independently fit the public examples using official isclose."""
    examples, query_t = _gravity_prompt_parts(str(sample.get("prompt", "")))
    fit = _gravity_verifier_fit(examples)
    answer = str(sample.get("answer", ""))
    example_checks = []
    if fit["valid"]:
        for time_value, expected_distance in examples:
            predicted_distance = _gravity_verifier_distance(float(time_value), float(fit["g_ols"]))
            example_checks.append(official_numeric_match(predicted_distance, expected_distance))
    answer_prediction = None
    if fit["valid"] and query_t is not None:
        answer_prediction = _gravity_verifier_distance(float(query_t), float(fit["g_ols"]))
    answer_ok = bool(fit["valid"] and example_checks and all(example_checks) and answer_prediction is not None and official_numeric_match(answer, answer_prediction))
    trace = sample.get("trace", {})
    trace_ok = isinstance(trace, dict) and trace.get("solver") == "ols_from_public_examples"
    if trace_ok:
        trace_ok = trace.get("query_t") == query_t and official_numeric_match(trace.get("g_ols"), fit.get("g_ols"))
        trace_ok = trace_ok and len(trace.get("g_estimates", [])) == len(examples)
    ident = gravity_identifiability(str(sample.get("prompt", "")))
    return {
        "answer_ok": answer_ok,
        "trace_ok": bool(trace_ok),
        "identifiability": ident,
        "example_checks": example_checks,
        "query_prediction": answer_prediction,
        "difficulty_ok": str(sample.get("difficulty", "")).startswith("examples_"),
        "solver_status": "independent_ols_verifier",
    }


In [ ]:
# Bit Manipulation: one-shot generator plus independent solver/verifier paths.
MASK8 = 0xFF
BIT_UNARY_OPS = ("NOT",)
BIT_SHIFT_OPS = ("SHL", "SHR", "ROTL", "ROTR")
BIT_BINARY_OPS = ("AND", "OR", "XOR")


# Generator-side implementation used only to materialize examples from a hidden tree.
def compute_bit_tree(node: tuple) -> int:
    kind = node[0]
    if kind == "lit":
        return int(node[1]) & MASK8
    if kind == "var":
        raise ValueError("uninstantiated_variable")
    if kind in BIT_SHIFT_OPS:
        value, shift = compute_bit_tree(node[1]), int(node[2]) % 8
        if kind == "SHL": return (value << shift) & MASK8
        if kind == "SHR": return (value >> shift) & MASK8
        if kind == "ROTL": return ((value << shift) | (value >> (8 - shift))) & MASK8 if shift else value
        return ((value >> shift) | (value << (8 - shift))) & MASK8 if shift else value
    if kind == "NOT":
        return (~compute_bit_tree(node[1])) & MASK8
    left, right = compute_bit_tree(node[1]), compute_bit_tree(node[2])
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def render_bit_tree(node: tuple) -> str:
    kind = node[0]
    if kind == "var": return "x"
    if kind == "lit": return f"0b{int(node[1]) & MASK8:08b}"
    if kind in BIT_SHIFT_OPS: return f"{kind}({render_bit_tree(node[1])}, {int(node[2])})"
    if kind == "NOT": return f"NOT({render_bit_tree(node[1])})"
    return f"{kind}({render_bit_tree(node[1])}, {render_bit_tree(node[2])})"


def instantiate_bit_tree(tree: tuple, value: int) -> tuple:
    """Instantiate every variable recursively, including both binary subtrees."""
    if tree[0] == "var":
        return ("lit", int(value) & MASK8)
    if tree[0] == "lit":
        return ("lit", int(tree[1]) & MASK8)
    if tree[0] in BIT_UNARY_OPS:
        return (tree[0], instantiate_bit_tree(tree[1], value))
    if tree[0] in BIT_SHIFT_OPS:
        return (tree[0], instantiate_bit_tree(tree[1], value), int(tree[2]) % 8)
    return (tree[0], instantiate_bit_tree(tree[1], value), instantiate_bit_tree(tree[2], value))


def random_bit_tree(rng: np.random.Generator, depth: int = 2) -> tuple:
    if depth <= 0:
        return ("var",)
    kind = str(rng.choice(BIT_SHIFT_OPS + BIT_BINARY_OPS + BIT_UNARY_OPS))
    left = random_bit_tree(rng, depth - 1)
    if kind in BIT_UNARY_OPS:
        return (kind, left)
    if kind in BIT_SHIFT_OPS:
        return (kind, left, int(rng.integers(0, 8)))
    return (kind, left, random_bit_tree(rng, depth - 1))


def _bit_tree_value(tree: tuple, value: int) -> int:
    return compute_bit_tree(instantiate_bit_tree(tree, value))


@lru_cache(maxsize=None)
def _bit_candidate_trees(max_depth: int = 2) -> tuple[tuple, ...]:
    by_depth = {0: (("var",),)}
    for depth in range(1, max_depth + 1):
        smaller = tuple(tree for d in range(depth) for tree in by_depth[d])
        candidates = []
        candidates.extend(("NOT", child) for child in smaller)
        candidates.extend((op, child, shift) for op in BIT_SHIFT_OPS for child in smaller for shift in range(8))
        candidates.extend((op, left, right) for op in BIT_BINARY_OPS for left in smaller for right in smaller)
        unique = {render_bit_tree(tree): tree for tree in candidates}
        by_depth[depth] = tuple(unique.values())
    all_trees = {render_bit_tree(tree): tree for depth in by_depth.values() for tree in depth}
    return tuple(all_trees.values())


# Solver-side evaluator is separate from the Generator's instantiate/compute path.
def _bit_solver_value(node: tuple, value: int) -> int:
    kind = node[0]
    if kind == "var": return int(value) & MASK8
    if kind == "lit": return int(node[1]) & MASK8
    if kind in BIT_SHIFT_OPS:
        child, shift = _bit_solver_value(node[1], value), int(node[2]) % 8
        if kind == "SHL": return (child << shift) & MASK8
        if kind == "SHR": return (child >> shift) & MASK8
        if kind == "ROTL": return ((child << shift) | (child >> (8 - shift))) & MASK8 if shift else child
        return ((child >> shift) | (child << (8 - shift))) & MASK8 if shift else child
    if kind == "NOT": return (~_bit_solver_value(node[1], value)) & MASK8
    left, right = _bit_solver_value(node[1], value), _bit_solver_value(node[2], value)
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def _bit_solver_tree_value(tree: tuple, value: int) -> int:
    return _bit_solver_value(tree, value)


def identify_bit_expression(example_pairs: list[tuple[int, int]], query_value: int, max_depth: int = 2) -> dict[str, object]:
    matching = []
    for tree in _bit_candidate_trees(max_depth):
        if all(_bit_solver_tree_value(tree, value) == output for value, output in example_pairs):
            matching.append(tree)
    semantic_groups = {}
    for tree in matching:
        signature = tuple(_bit_solver_tree_value(tree, value) for value in range(256))
        semantic_groups.setdefault(signature, tree)
    query_outputs = sorted({_bit_solver_tree_value(tree, query_value) for tree in semantic_groups.values()})
    selected_tree = next(iter(semantic_groups.values()), None)
    return {
        "raw_candidate_count": len(matching),
        "semantic_candidate_count": len(semantic_groups),
        "query_output_count": len(query_outputs),
        "query_outputs": [f"{value:08b}" for value in query_outputs],
        "expression_unique": len(matching) == 1,
        "answer_unique": len(query_outputs) == 1,
        "selected_expression": render_bit_tree(selected_tree) if selected_tree else None,
    }


def _parse_bit_expression(text: str, position: int = 0) -> tuple[tuple, int]:
    while position < len(text) and text[position].isspace(): position += 1
    if text.startswith("x", position): return ("var",), position + 1
    literal = re.match(r"0b([01]{8})", text[position:])
    if literal: return ("lit", int(literal.group(1), 2)), position + literal.end()
    operation = re.match(r"(NOT|SHL|SHR|ROTL|ROTR|AND|OR|XOR)\(", text[position:])
    if operation is None: raise ValueError("invalid_bit_expression")
    op, pos = operation.group(1), position + operation.end()
    left, pos = _parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if op == "NOT":
        if pos >= len(text) or text[pos] != ")": raise ValueError("invalid_not")
        return (op, left), pos + 1
    if pos >= len(text) or text[pos] != ",": raise ValueError("missing_comma")
    pos += 1
    if op in BIT_SHIFT_OPS:
        match = re.match(r"\s*([0-7])\s*\)", text[pos:])
        if match is None: raise ValueError("invalid_shift")
        return (op, left, int(match.group(1))), pos + match.end()
    right, pos = _parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if pos >= len(text) or text[pos] != ")": raise ValueError("missing_close")
    return (op, left, right), pos + 1


def _evaluate_bit_expression(text: str, value: int) -> int:
    tree, position = _parse_bit_expression(text)
    if text[position:].strip(): raise ValueError("trailing_expression")
    return _bit_solver_tree_value(tree, value)


def solve_bit_prompt(prompt: str) -> dict[str, object]:
    pairs = [(int(value, 2), int(output, 2)) for value, output in re.findall(r"Input: 0b([01]{8}) -> Output: 0b([01]{8})", str(prompt))]
    query_match = re.search(r"output for input: 0b([01]{8})", str(prompt))
    if not query_match or not pairs:
        return {"status": "invalid_prompt", "answer": None, "trace": {}, "identifiability": {"answer_unique": False}}
    query_value = int(query_match.group(1), 2)
    ident = identify_bit_expression(pairs, query_value, max_depth=2)
    if not ident["answer_unique"]:
        return {"status": "ambiguous", "answer": None, "trace": {"solver": "finite_semantic_enumeration", "identifiability": ident}, "identifiability": ident}
    answer = ident["query_outputs"][0]
    return {
        "status": "solved",
        "answer": answer,
        "trace": {"solver": "finite_semantic_enumeration", "expression": ident["selected_expression"], "example_inputs": [v for v, _ in pairs], "query_input": query_value, "identifiability": ident},
        "identifiability": ident,
    }


def generate_bit_sample(rng: np.random.Generator, examples: int = 5) -> dict[str, object]:
    """Generator: exactly one random DSL candidate; no identifiability retry loop."""
    tree = random_bit_tree(rng, depth=2)
    inputs = rng.choice(np.arange(256), size=examples + 1, replace=False).tolist()
    pairs = [(int(value), _bit_tree_value(tree, int(value))) for value in inputs[:-1]]
    query_value = int(inputs[-1])
    prompt = (
        "Below are input-output examples for a hidden deterministic 8-bit transformation.\n"
        + "\n".join(f"Input: 0b{value:08b} -> Output: 0b{output:08b}" for value, output in pairs)
        + f"\nNow, determine the output for input: 0b{query_value:08b}"
    )
    expression = render_bit_tree(tree)
    return {
        "task_type": "bit_manipulation",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"expression": expression, "depth": 2, "width": 8, "operations": sorted(set(re.findall(r"\b(?:SHL|SHR|ROTL|ROTR|AND|OR|XOR|NOT)\b", expression)))},
        "prompt": prompt,
        "answer": "",
        "trace": {},
        "difficulty": "depth_2",
        "deduce_or_guess": "deduce",
        "generation_status": "generated",
        "generation_error": None,
    }


# Verifier-side parser and evaluator are intentionally separate from Generator and Solver.
def _verify_bit_value(node: tuple, value: int) -> int:
    kind = node[0]
    if kind == "var": return int(value) & MASK8
    if kind == "lit": return int(node[1]) & MASK8
    if kind in BIT_SHIFT_OPS:
        child, shift = _verify_bit_value(node[1], value), int(node[2]) % 8
        if kind == "SHL": return (child << shift) & MASK8
        if kind == "SHR": return (child >> shift) & MASK8
        if kind == "ROTL": return ((child << shift) | (child >> (8 - shift))) & MASK8 if shift else child
        return ((child >> shift) | (child << (8 - shift))) & MASK8 if shift else child
    if kind == "NOT": return (~_verify_bit_value(node[1], value)) & MASK8
    left, right = _verify_bit_value(node[1], value), _verify_bit_value(node[2], value)
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def _verify_parse_bit_expression(text: str, position: int = 0) -> tuple[tuple, int]:
    while position < len(text) and text[position].isspace(): position += 1
    if text.startswith("x", position): return ("var",), position + 1
    literal = re.match(r"0b([01]{8})", text[position:])
    if literal: return ("lit", int(literal.group(1), 2)), position + literal.end()
    operation = re.match(r"(NOT|SHL|SHR|ROTL|ROTR|AND|OR|XOR)\(", text[position:])
    if operation is None: raise ValueError("invalid_bit_expression")
    op, pos = operation.group(1), position + operation.end()
    left, pos = _verify_parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if op == "NOT":
        if pos >= len(text) or text[pos] != ")": raise ValueError("invalid_not")
        return (op, left), pos + 1
    if pos >= len(text) or text[pos] != ",": raise ValueError("missing_comma")
    pos += 1
    if op in BIT_SHIFT_OPS:
        match = re.match(r"\s*([0-7])\s*\)", text[pos:])
        if match is None: raise ValueError("invalid_shift")
        return (op, left, int(match.group(1))), pos + match.end()
    right, pos = _verify_parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if pos >= len(text) or text[pos] != ")": raise ValueError("missing_close")
    return (op, left, right), pos + 1


def _verify_evaluate_bit_expression(text: str, value: int) -> int:
    tree, position = _verify_parse_bit_expression(text)
    if text[position:].strip(): raise ValueError("trailing_expression")
    return _verify_bit_value(tree, value)


def _verify_bit_identifiability(example_pairs: list[tuple[int, int]], query_value: int, max_depth: int = 2) -> dict[str, object]:
    matching = []
    for tree in _bit_candidate_trees(max_depth):
        if all(_verify_bit_value(tree, value) == output for value, output in example_pairs):
            matching.append(tree)
    semantic_groups = {}
    for tree in matching:
        signature = tuple(_verify_bit_value(tree, value) for value in range(256))
        semantic_groups.setdefault(signature, tree)
    query_outputs = sorted({_verify_bit_value(tree, query_value) for tree in semantic_groups.values()})
    return {
        "raw_candidate_count": len(matching),
        "semantic_candidate_count": len(semantic_groups),
        "query_output_count": len(query_outputs),
        "query_outputs": [f"{value:08b}" for value in query_outputs],
        "expression_unique": len(matching) == 1,
        "answer_unique": len(query_outputs) == 1,
    }


def verify_bit_sample(sample: dict[str, object]) -> dict[str, object]:
    prompt = str(sample.get("prompt", ""))
    pairs = [(int(value, 2), int(output, 2)) for value, output in re.findall(r"Input: 0b([01]{8}) -> Output: 0b([01]{8})", prompt)]
    query_match = re.search(r"output for input: 0b([01]{8})", prompt)
    trace = sample.get("trace", {})
    answer = str(sample.get("answer", "")).removeprefix("0b")
    ident = _verify_bit_identifiability(pairs, int(query_match.group(1), 2), max_depth=2) if query_match else {"query_output_count": 0, "answer_unique": False, "expression_unique": False, "query_outputs": []}
    answer_ok = bool(query_match and pairs and re.fullmatch(r"[01]{8}", answer) and ident.get("answer_unique") and answer in ident.get("query_outputs", []))
    trace_ok = isinstance(trace, dict) and trace.get("solver") == "finite_semantic_enumeration"
    if answer_ok:
        try:
            expression = str(trace["expression"])
            answer_ok = all(_verify_evaluate_bit_expression(expression, value) == output for value, output in pairs)
            answer_ok = answer_ok and _verify_evaluate_bit_expression(expression, int(query_match.group(1), 2)) == int(answer, 2)
            trace_ok = trace_ok and trace.get("example_inputs") == [v for v, _ in pairs] and trace.get("query_input") == int(query_match.group(1), 2)
        except (KeyError, ValueError, TypeError, IndexError):
            answer_ok = False
            trace_ok = False
    return {"answer_ok": bool(answer_ok), "trace_ok": bool(trace_ok), "identifiability": ident, "difficulty_ok": str(sample.get("difficulty", "")).startswith("depth_"), "solver_status": "independent_parser_and_finite_enumeration"}


In [ ]:
# Equation Numeric: 24-rule prior, independent solver, and deterministic verifier.
NUMERIC_EXPRESSION_RE = re.compile(r"^\d{2}[^\d]\d{2}$")
NUMERIC_OPERATOR_POOL = TRAIN_NUMERIC_OPERATOR_POOL or tuple("+-*/|\\@$%&!?^#~`")


def _numeric_operands(expression: str, parse_name: str) -> tuple[str, str]:
    if NUMERIC_EXPRESSION_RE.fullmatch(expression) is None:
        raise ValueError("invalid_numeric_expression")
    left_text, right_text = expression[:2], expression[3:]
    if parse_name == "AB_CD":
        return left_text, right_text
    if parse_name == "BA_DC":
        return left_text[::-1], right_text[::-1]
    raise ValueError("unsupported_parse")


def _numeric_operation(operation: str, left_text: str, right_text: str) -> int | str | None:
    left, right = int(left_text), int(right_text)
    if operation == "concat_lr": return left_text + right_text
    if operation == "concat_rl": return right_text + left_text
    if operation == "add": return left + right
    if operation == "add_p1": return left + right + 1
    if operation == "add_m1": return left + right - 1
    if operation == "sub_lr": return left - right
    if operation == "sub_rl": return right - left
    if operation == "abs_sub": return abs(left - right)
    if operation == "mul": return left * right
    if operation == "mul_p1": return left * right + 1
    if operation == "mul_m1": return left * right - 1
    if operation == "max_mod_min":
        smaller = min(left, right)
        return None if smaller == 0 else max(left, right) % smaller
    raise ValueError(f"unsupported_operation:{operation}")


def _numeric_sign_token(operator: str) -> str:
    return "-" if operator == "-" else operator


def _numeric_format(value: int | str, format_name: str, operator: str) -> str | None:
    if format_name == "raw":
        if isinstance(value, str): return value
        return str(value) if value >= 0 else f"{_numeric_sign_token(operator)}{abs(value)}"
    if not isinstance(value, int): return None
    if format_name == "rev": return str(value)[::-1] if value >= 0 else None
    if format_name == "sign_prefix_rev": return str(value)[::-1] if value >= 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "zpad2": return f"{_numeric_sign_token(operator)}{abs(value):02d}" if value < 0 else f"{value:02d}"
    if format_name == "neg_prefix_rev": return "0" if value == 0 else (None if value > 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}")
    if format_name == "always_prefix_rev": return "0" if value == 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "sign_prefix_raw": return "0" if value == 0 else f"{operator}{abs(value)}"
    if format_name == "sign_prefix_zpad2": return "00" if value == 0 else f"{operator}{abs(value):02d}"
    if format_name == "neg_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "always_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "rev_or_neg_suffix": return str(value)[::-1] if value >= 0 else f"{str(abs(value))[::-1]}{operator}"
    raise ValueError(f"unsupported_format:{format_name}")


def numeric_rule_output(rule_name: str, expression: str) -> str | None:
    spec = NUMERIC24_RULE_BY_NAME[rule_name]
    left_text, right_text = _numeric_operands(expression, spec["parse"])
    operation_value = _numeric_operation(spec["operation"], left_text, right_text)
    return None if operation_value is None else _numeric_format(operation_value, spec["format"], expression[2])


def _parse_numeric_prompt(prompt: str) -> tuple[list[tuple[str, str]], str] | None:
    examples, query = [], None
    for line in str(prompt).splitlines():
        line = line.strip()
        if line.startswith("Now, determine the result for:"):
            query = line.split(":", 1)[1].strip()
        elif " = " in line:
            expression, output = line.split(" = ", 1)
            if NUMERIC_EXPRESSION_RE.fullmatch(expression.strip()): examples.append((expression.strip(), output.strip()))
    return (examples, query) if examples and query else None


def _numeric_candidate_rules(examples: list[tuple[str, str]], operator: str) -> list[str]:
    same_operator = [(expression, output) for expression, output in examples if expression[2] == operator]
    candidates = []
    for rule in NUMERIC24_RULES:
        if all(numeric_or_text_match(numeric_rule_output(rule["name"], expression), output) for expression, output in same_operator):
            candidates.append(rule["name"])
    return candidates


def _numeric_valid_assignments(candidates_by_operator: dict[str, list[str]], operators: list[str]) -> list[dict[str, str]]:
    assignments = []
    for mode in ("normal", "flip"):
        selected = {}
        used_groups = set()
        def backtrack(index: int) -> None:
            if index == len(operators):
                assignments.append({"mode": mode, "rules": dict(selected)})
                return
            operator = operators[index]
            for rule_name in candidates_by_operator[operator]:
                spec = NUMERIC24_RULE_BY_NAME[rule_name]
                if spec["family"] != "join" and spec["family"] != mode:
                    continue
                group = spec["group"]
                if group != "join_like" and group in used_groups:
                    continue
                selected[operator] = rule_name
                if group != "join_like": used_groups.add(group)
                backtrack(index + 1)
                if group != "join_like": used_groups.remove(group)
                selected.pop(operator, None)
        backtrack(0)
    return assignments


def solve_equation_numeric_prompt(prompt: str) -> dict[str, object]:
    parsed = _parse_numeric_prompt(prompt)
    if parsed is None: return {"status": "invalid_prompt", "query_outputs": []}
    examples, query_expression = parsed
    query_operator = query_expression[2]
    example_operators = []
    for expression, _ in examples:
        if expression[2] not in example_operators: example_operators.append(expression[2])
    operators = example_operators + ([] if query_operator in example_operators else [query_operator])
    candidates_by_operator = {operator: _numeric_candidate_rules(examples, operator) for operator in operators}
    if any(not values for values in candidates_by_operator.values()):
        return {"status": "no_compatible_rule", "query_outputs": [], "candidates_by_operator": candidates_by_operator}
    assignments = _numeric_valid_assignments(candidates_by_operator, operators)
    query_outputs = []
    for assignment in assignments:
        value = numeric_rule_output(assignment["rules"][query_operator], query_expression)
        if value is not None and not any(numeric_or_text_match(value, previous) for previous in query_outputs):
            query_outputs.append(value)
    query_outputs.sort()
    selected = assignments[0] if assignments else None
    return {
        "status": "solved" if assignments else "no_valid_global_assignment",
        "query_operator": query_operator,
        "query_operator_in_examples": query_operator in example_operators,
        "candidates_by_operator": candidates_by_operator,
        "valid_assignment_count": len(assignments),
        "valid_assignments": assignments[:20],
        "query_outputs": query_outputs,
        "answer_unique": len(query_outputs) == 1,
        "selected_assignment": selected,
    }


def _numeric_truth_output(rule_name: str, expression: str) -> str | None:
    """Generator-side arithmetic implementation kept separate from the solver API."""
    spec = NUMERIC24_RULE_BY_NAME[rule_name]
    left, right = expression[:2], expression[3:]
    if spec["parse"] == "BA_DC": left, right = left[::-1], right[::-1]
    a, b = int(left), int(right)
    operation = spec["operation"]
    if operation == "concat_lr": value = left + right
    elif operation == "concat_rl": value = right + left
    elif operation == "add": value = a + b
    elif operation == "add_p1": value = a + b + 1
    elif operation == "add_m1": value = a + b - 1
    elif operation == "sub_lr": value = a - b
    elif operation == "sub_rl": value = b - a
    elif operation == "abs_sub": value = abs(a - b)
    elif operation == "mul": value = a * b
    elif operation == "mul_p1": value = a * b + 1
    elif operation == "mul_m1": value = a * b - 1
    elif operation == "max_mod_min":
        value = None if min(a, b) == 0 else max(a, b) % min(a, b)
    else: raise ValueError(operation)
    return None if value is None else _numeric_format(value, spec["format"], expression[2])


def _numeric_random_expression(rng: np.random.Generator, operator: str, avoid_zero_divisor: bool = False) -> str:
    left, right = int(rng.integers(0, 100)), int(rng.integers(0, 100))
    if avoid_zero_divisor and right == 0: right = int(rng.integers(1, 100))
    return f"{left:02d}{operator}{right:02d}"


def _numeric_prompt_text(example_pairs: list[tuple[str, str]], query_expression: str) -> str:
    return (
        "In this numeric equation task, each operator has a hidden transformation rule.\n"
        "The same rule assignment is used consistently within this prompt.\n"
        + "\n".join(f"{expression} = {output}" for expression, output in example_pairs)
        + f"\nNow, determine the result for: {query_expression}"
    )


def _numeric_hidden_assignment(rng: np.random.Generator, mode: str) -> dict[str, str]:
    pools = {}
    for group in ("sub_or_mod_like", "add_like", "mul_like"):
        choices = [rule["name"] for rule in NUMERIC24_RULES if rule["family"] == mode and rule["group"] == group]
        pools[group] = str(rng.choice(choices))
    pools["join_like"] = str(rng.choice(["join", "join_swap"]))
    return pools


def generate_equation_numeric_sample(
    rng: np.random.Generator,
    guess: bool = False,
    guess_variant: str | None = None,
) -> dict[str, object]:
    """Generator: exactly one multi-operator candidate, with no retry loop."""
    mode = str(rng.choice(["normal", "flip"]))
    by_group = _numeric_hidden_assignment(rng, mode)
    if guess:
        example_operators = list(rng.choice(np.asarray(NUMERIC_OPERATOR_POOL), size=3, replace=False))
        query_operator = next(operator for operator in NUMERIC_OPERATOR_POOL if operator not in example_operators)
        hidden = {operator: by_group[group] for operator, group in zip(example_operators, ("sub_or_mod_like", "add_like", "mul_like"))}
        hidden[query_operator] = by_group["join_like"]
    else:
        operators = list(rng.choice(np.asarray(NUMERIC_OPERATOR_POOL), size=4, replace=False))
        hidden = {operator: by_group[group] for operator, group in zip(operators, ("sub_or_mod_like", "add_like", "mul_like", "join_like"))}
        query_operator = str(rng.choice(np.asarray(operators)))
        example_operators = operators
    pairs = []
    for operator in example_operators:
        rule_name = hidden[operator]
        for _example in range(5):
            expression = _numeric_random_expression(rng, operator, avoid_zero_divisor="mod" in NUMERIC24_RULE_BY_NAME[rule_name]["operation"])
            output = _numeric_truth_output(rule_name, expression)
            if output is not None:
                pairs.append((expression, output))
    if guess and guess_variant == "guess_answer_equivalent":
        same = int(rng.integers(10, 100))
        query_expression = f"{same:02d}{query_operator}{same:02d}"
    elif guess:
        left = int(rng.integers(10, 100))
        right = int(rng.integers(10, 100))
        if right == left:
            right = (right + 1) % 90 + 10
        query_expression = f"{left:02d}{query_operator}{right:02d}"
    else:
        query_expression = _numeric_random_expression(rng, query_operator, avoid_zero_divisor=False)
    return {
        "task_type": "equation_numeric",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"mode": mode, "operator_to_rule": hidden, "rule_formulas": {name: NUMERIC24_FORMULAS[name] for name in hidden.values()}},
        "prompt": _numeric_prompt_text(pairs, query_expression),
        "answer": "",
        "trace": {},
        "difficulty": "guess_candidate" if guess else "deduce_multi_operator",
        "deduce_or_guess": "guess" if guess else "deduce",
        "guess_subtype": "pending" if guess else "deduce",
        "generation_status": "generated",
        "generation_error": None,
    }


def _numeric_verifier_format(value: int | str, format_name: str, operator: str) -> str | None:
    """Verifier-only formatter; kept separate from generator and Solver formatters."""
    if format_name == "raw":
        if isinstance(value, str): return value
        return str(value) if value >= 0 else f"{_numeric_sign_token(operator)}{abs(value)}"
    if not isinstance(value, int): return None
    if format_name == "rev": return str(value)[::-1] if value >= 0 else None
    if format_name == "sign_prefix_rev": return str(value)[::-1] if value >= 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "zpad2": return f"{_numeric_sign_token(operator)}{abs(value):02d}" if value < 0 else f"{value:02d}"
    if format_name == "neg_prefix_rev": return "0" if value == 0 else (None if value > 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}")
    if format_name == "always_prefix_rev": return "0" if value == 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "sign_prefix_raw": return "0" if value == 0 else f"{operator}{abs(value)}"
    if format_name == "sign_prefix_zpad2": return "00" if value == 0 else f"{operator}{abs(value):02d}"
    if format_name == "neg_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "always_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "rev_or_neg_suffix": return str(value)[::-1] if value >= 0 else f"{str(abs(value))[::-1]}{operator}"
    raise ValueError(f"unsupported_format:{format_name}")


def _numeric_verify_output(rule_name: str, expression: str) -> str | None:
    """Verifier-side arithmetic and formatting implementation."""
    spec = NUMERIC24_RULE_BY_NAME.get(rule_name)
    if spec is None or NUMERIC_EXPRESSION_RE.fullmatch(expression) is None:
        return None
    left_text, right_text = expression[:2], expression[3:]
    if spec["parse"] == "BA_DC": left_text, right_text = left_text[::-1], right_text[::-1]
    left, right = int(left_text), int(right_text)
    op = spec["operation"]
    if op == "concat_lr": value = left_text + right_text
    elif op == "concat_rl": value = right_text + left_text
    elif op == "add": value = left + right
    elif op == "add_p1": value = left + right + 1
    elif op == "add_m1": value = left + right - 1
    elif op == "sub_lr": value = left - right
    elif op == "sub_rl": value = right - left
    elif op == "abs_sub": value = abs(left - right)
    elif op == "mul": value = left * right
    elif op == "mul_p1": value = left * right + 1
    elif op == "mul_m1": value = left * right - 1
    elif op == "max_mod_min": value = None if min(left, right) == 0 else max(left, right) % min(left, right)
    else: return None
    return None if value is None else _numeric_verifier_format(value, spec["format"], expression[2])


def _numeric_verifier_candidate_rules(examples: list[tuple[str, str]], operator: str) -> list[str]:
    same_operator = [(expression, output) for expression, output in examples if expression[2] == operator]
    candidates = []
    for rule in NUMERIC24_RULES:
        if all(numeric_or_text_match(_numeric_verify_output(rule["name"], expression), output) for expression, output in same_operator):
            candidates.append(rule["name"])
    return candidates


def _numeric_verifier_valid_assignments(candidates_by_operator: dict[str, list[str]], operators: list[str]) -> list[dict[str, str]]:
    assignments = []
    for mode in ("normal", "flip"):
        selected, used_groups = {}, set()
        def backtrack(index: int) -> None:
            if index == len(operators):
                assignments.append({"mode": mode, "rules": dict(selected)})
                return
            operator = operators[index]
            for rule_name in candidates_by_operator.get(operator, []):
                spec = NUMERIC24_RULE_BY_NAME[rule_name]
                if spec["family"] != "join" and spec["family"] != mode:
                    continue
                group = spec["group"]
                if group != "join_like" and group in used_groups:
                    continue
                selected[operator] = rule_name
                if group != "join_like": used_groups.add(group)
                backtrack(index + 1)
                if group != "join_like": used_groups.remove(group)
                selected.pop(operator, None)
        backtrack(0)
    return assignments


def verify_equation_numeric_prompt(prompt: str) -> dict[str, object]:
    """Independent Verifier path; does not call solve_equation_numeric_prompt."""
    parsed = _parse_numeric_prompt(prompt)
    if parsed is None:
        return {"status": "invalid_prompt", "query_outputs": [], "valid_assignment_count": 0}
    examples, query_expression = parsed
    query_operator = query_expression[2]
    example_operators = []
    for expression, _ in examples:
        if expression[2] not in example_operators:
            example_operators.append(expression[2])
    operators = example_operators + ([] if query_operator in example_operators else [query_operator])
    candidates_by_operator = {operator: _numeric_verifier_candidate_rules(examples, operator) for operator in operators}
    if any(not values for values in candidates_by_operator.values()):
        return {"status": "no_compatible_rule", "query_outputs": [], "candidates_by_operator": candidates_by_operator, "valid_assignment_count": 0}
    assignments = _numeric_verifier_valid_assignments(candidates_by_operator, operators)
    query_outputs = []
    for assignment in assignments:
        value = _numeric_verify_output(assignment["rules"][query_operator], query_expression)
        if value is not None and not any(numeric_or_text_match(value, previous) for previous in query_outputs):
            query_outputs.append(value)
    query_outputs.sort()
    return {
        "status": "solved" if assignments else "no_valid_global_assignment",
        "query_operator": query_operator,
        "query_operator_in_examples": query_operator in example_operators,
        "candidates_by_operator": candidates_by_operator,
        "valid_assignment_count": len(assignments),
        "valid_assignments": assignments[:20],
        "query_outputs": query_outputs,
        "answer_unique": len(query_outputs) == 1,
        "selected_assignment": assignments[0] if assignments else None,
    }


def verify_equation_numeric_sample(sample: dict[str, object]) -> dict[str, object]:
    parsed = _parse_numeric_prompt(str(sample.get("prompt", "")))
    if parsed is None:
        return {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False, "guess_subtype": "invalid"}
    examples, query_expression = parsed
    answer = str(sample.get("answer", ""))
    resolution = verify_equation_numeric_prompt(str(sample.get("prompt", "")))
    ident = {
        "valid_assignment_count": resolution.get("valid_assignment_count", 0),
        "query_output_count": len(resolution.get("query_outputs", [])),
        "query_outputs": resolution.get("query_outputs", []),
        "answer_unique": bool(resolution.get("answer_unique", False)),
        "rule_assignment_unique": resolution.get("valid_assignment_count", 0) == 1,
    }
    if query_expression[2] not in {expression[2] for expression, _ in examples}:
        if ident["rule_assignment_unique"]:
            guess_subtype = "guess_rule_unique"
        elif ident["answer_unique"]:
            guess_subtype = "guess_answer_equivalent"
        else:
            guess_subtype = "guess_answer_ambiguous"
    else:
        guess_subtype = "deduce"
    trace = sample.get("trace", {})
    trace_ok = isinstance(trace, dict) and trace.get("query_outputs") == resolution.get("query_outputs")
    answer_ok = bool(resolution.get("answer_unique", False) and any(numeric_or_text_match(answer, output) for output in resolution.get("query_outputs", [])))
    if guess_subtype == "guess_answer_ambiguous":
        answer_ok = False
    return {
        "answer_ok": answer_ok,
        "trace_ok": bool(trace_ok),
        "identifiability": ident,
        "guess_subtype": guess_subtype,
        "difficulty_ok": str(sample.get("difficulty", "")).startswith(("deduce_", "guess_")),
        "solver_status": "independent_24_rule_verifier",
    }


In [ ]:
SYNTHETIC_VERIFIER_VERSION = "synthetic_independent_verifier_v3"

# Unified screening, export, leakage audit, and optional local-vLLM Data Designer Preview.
import copy

SYNTHETIC_SCHEMA_FIELDS = [
    "candidate_id", "task_type", "source", "generator_version", "hidden_rule", "prompt", "answer", "trace", "difficulty",
    "deduce_or_guess", "generation_status", "generation_error",
]


def synthetic_prompt_fingerprint(text: object) -> str:
    normalized = re.sub(r"\s+", " ", str(text).lower().strip())
    return prompt_hash(normalized)


def structure_fingerprint(text: object, task_type: str) -> str:
    """Mask literal values while preserving task words, operators, and expression shape."""
    normalized = re.sub(r"\s+", " ", str(text).lower().strip())
    normalized = re.sub(r"0b[01]+", "<bit>", normalized)
    if task_type == "equation_numeric":
        normalized = re.sub(r"\d{2}", "<2d>", normalized)
    else:
        normalized = re.sub(r"[-+]?\d+(?:\.\d+)?", "<num>", normalized)
    return prompt_hash(normalized)


def _screen_verification(sample: dict[str, object]) -> dict[str, object]:
    if sample.get("task_type") == "gravity": return verify_gravity_sample(sample)
    if sample.get("task_type") == "bit_manipulation": return verify_bit_sample(sample)
    if sample.get("task_type") == "equation_numeric": return verify_equation_numeric_sample(sample)
    return {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False}


def screen_synthetic_sample(sample: dict[str, object], seen_exact: set[str], seen_normalized: set[str], seen_structure: set[str]) -> dict[str, object]:
    reasons = []
    required_fields = [field for field in SYNTHETIC_SCHEMA_FIELDS if field != "generation_error"]
    schema_ok = all(field in sample for field in required_fields) and "generation_error" in sample
    schema_ok = schema_ok and isinstance(sample.get("prompt"), str) and isinstance(sample.get("trace"), dict)
    if not schema_ok: reasons.append("schema_invalid")
    if sample.get("generation_status") != "generated": reasons.append("generation_failed")
    if sample.get("task_type") not in TARGET_TASKS: reasons.append("unsupported_task")
    if sample.get("source") != "deterministic_prior_generator": reasons.append("unexpected_source")
    exact_key = prompt_hash(sample.get("prompt", ""))
    normalized_key = synthetic_prompt_fingerprint(sample.get("prompt", ""))
    structure_key = structure_fingerprint(sample.get("prompt", ""), str(sample.get("task_type", "")))
    if exact_key in seen_exact: reasons.append("duplicate_prompt")
    if normalized_key in seen_normalized: reasons.append("near_duplicate_prompt")
    structure_duplicate = structure_key in seen_structure
    verification = _screen_verification(sample)
    if not verification.get("answer_ok", False): reasons.append("answer_verification_failed")
    if not verification.get("trace_ok", False): reasons.append("trace_verification_failed")
    ident = verification.get("identifiability", {})
    if not ident.get("answer_unique", False): reasons.append("answer_not_identifiable")
    if not verification.get("difficulty_ok", False): reasons.append("difficulty_audit_failed")
    guess_subtype = verification.get("guess_subtype", sample.get("guess_subtype", "deduce"))
    if sample.get("task_type") == "equation_numeric":
        sample["guess_subtype"] = guess_subtype
        if guess_subtype == "guess_answer_ambiguous": reasons.append("guess_answer_ambiguous")
    accepted = not reasons
    record = {
        "candidate_id": sample.get("candidate_id", sample.get("candidate_index", "")),
        "candidate_index": sample.get("candidate_index", -1),
        "task_type": sample.get("task_type", ""),
        "source": sample.get("source", ""),
        "generator_version": sample.get("generator_version", ""),
        "generation_status": sample.get("generation_status", ""),
        "generation_error": sample.get("generation_error"),
        "deduce_or_guess": sample.get("deduce_or_guess", "unknown"),
        "guess_subtype": guess_subtype,
        "prompt_hash": exact_key,
        "normalized_prompt_hash": normalized_key,
        "structure_fingerprint": structure_key,
        "structure_duplicate_within_synthetic": structure_duplicate,
        "schema_ok": schema_ok,
        "answer_ok": bool(verification.get("answer_ok", False)),
        "trace_ok": bool(verification.get("trace_ok", False)),
        "identifiability": json.dumps(ident, ensure_ascii=False, sort_keys=True),
        "difficulty": sample.get("difficulty", ""),
        "accepted": accepted,
        "verification_status": "accepted" if accepted else "rejected",
        "verifier_version": SYNTHETIC_VERIFIER_VERSION,
        "rejection_reasons": ";".join(dict.fromkeys(reasons)),
    }
    return record


def _write_jsonl(path: Path, records: list[dict[str, object]]) -> None:
    payload = "\n".join(json.dumps(record, ensure_ascii=False) for record in records)
    path.write_text(payload + ("\n" if payload else ""), encoding="utf-8")


def _real_synthetic_cross_audit(candidates: list[dict[str, object]]) -> dict[str, object]:
    """Compare synthetic prompts with all real splits; structure matches are risks, not leakage claims."""
    real_index = {"exact": {}, "normalized": {}, "structure": {}}
    for row in split_df.itertuples(index=False):
        exact = prompt_hash(row.prompt)
        normalized = prompt_hash(normalize_prompt(row.prompt))
        structure = structure_fingerprint(row.prompt, row.task_type)
        for key, value in [("exact", exact), ("normalized", normalized), ("structure", structure)]:
            real_index[key].setdefault(value, []).append({"split": row.split, "id": row.id, "source_row": int(row.source_row), "task_type": row.task_type})
    rows = []
    for sample in candidates:
        task_type = str(sample.get("task_type", ""))
        keys = {
            "exact": prompt_hash(sample.get("prompt", "")),
            "normalized": prompt_hash(normalize_prompt(sample.get("prompt", ""))),
            "structure": structure_fingerprint(sample.get("prompt", ""), task_type),
        }
        matches = {kind: real_index[kind].get(key, []) for kind, key in keys.items()}
        rows.append({
            "candidate_id": sample.get("candidate_id", ""),
            "task_type": task_type,
            "source": sample.get("source", ""),
            "exact_prompt_match": bool(matches["exact"]),
            "normalized_prompt_match": bool(matches["normalized"]),
            "structure_match": bool(matches["structure"]),
            "exact_matches": matches["exact"],
            "normalized_matches": matches["normalized"],
            "structure_matches": matches["structure"],
            "interpretation": "structure_match_is_risk_signal_only",
        })
    audit_df = pd.DataFrame(rows)
    audit_df.to_csv(SYNTHETIC_DIR / "synthetic_real_leakage_audit.csv", index=False)
    summary = {
        "candidate_count": len(rows),
        "exact_prompt_matches": int(audit_df["exact_prompt_match"].sum()) if len(audit_df) else 0,
        "normalized_prompt_matches": int(audit_df["normalized_prompt_match"].sum()) if len(audit_df) else 0,
        "structure_matches": int(audit_df["structure_match"].sum()) if len(audit_df) else 0,
        "by_task": {
            task: {
                "candidates": int(len(group)),
                "exact_prompt_matches": int(group["exact_prompt_match"].sum()),
                "normalized_prompt_matches": int(group["normalized_prompt_match"].sum()),
                "structure_matches": int(group["structure_match"].sum()),
            }
            for task, group in audit_df.groupby("task_type", sort=True)
        },
        "source_split_counts": {split: int((split_df["split"] == split).sum()) for split in sorted(split_df["split"].unique())},
        "limitations": [
            "Synthetic generation uses only the real Train split profile and does not read Validation/Holdout answers.",
            "Normalized and structure matches are risk indicators; they do not prove same-source leakage.",
            "No original source_row is attached to deterministic synthetic candidates, so augmentation lineage is recorded as unavailable.",
    ]}
    (SYNTHETIC_DIR / "synthetic_real_leakage_audit.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return summary


def _data_designer_preview() -> dict[str, object]:
    """Optional official Data Designer API using only a local OpenAI-compatible vLLM endpoint."""
    if not RUN_DATA_DESIGNER_PREVIEW:
        return {"status": "skipped_by_default", "reason": "RUN_DATA_DESIGNER_PREVIEW=False"}
    endpoint = os.environ.get("LOCAL_VLLM_ENDPOINT", "http://127.0.0.1:8000/v1")
    served_model = os.environ.get("LOCAL_VLLM_MODEL", "Qwen3-4B-Thinking-2507")
    try:
        import data_designer.config as dd
        from data_designer.interface import DataDesigner
    except ImportError as exc:
        return {"status": "blocked_missing_data_designer", "reason": repr(exc), "install": "按官方 Data Designer 文档安装；Notebook 不临时安装依赖"}
    try:
        accepted = [json.loads(line) for line in (SYNTHETIC_DIR / "synthetic_accepted.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
        seed_rows = []
        for task_type in ("gravity", "equation_numeric", "bit_manipulation"):
            task_rows = [row for row in accepted if row.get("task_type") == task_type]
            if len(task_rows) < 3:
                return {"status": "blocked_insufficient_stratified_seed", "task_type": task_type, "available": len(task_rows)}
            seed_rows.extend({"task_type": row["task_type"], "prompt": row["prompt"]} for row in task_rows[:3])
        seed_path = SYNTHETIC_DIR / "data_designer_seed.jsonl"
        _write_jsonl(seed_path, seed_rows)
        provider = dd.ModelProvider(name="local-vllm", endpoint=endpoint, provider_type="openai", api_key=None)
        model_configs = [dd.ModelConfig(
            alias="local-qwen3",
            model=served_model,
            provider="local-vllm",
            inference_parameters=dd.ChatCompletionInferenceParams(temperature=0.3, top_p=0.9, max_tokens=512),
        )]
        builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
        builder.with_seed_dataset(dd.LocalFileSeedSource(path=str(seed_path)))
        builder.add_column(dd.LLMTextColumnConfig(
            name="prompt_variant",
            model_alias="local-qwen3",
            prompt="Rewrite the task prompt while preserving every number, operator, input, and output example. Return only the rewritten prompt.\n{{ prompt }}",
        ))
        builder.add_column(dd.LLMTextColumnConfig(
            name="trace_variant",
            model_alias="local-qwen3",
            prompt="Write a candidate reasoning trace for this task. Do not invent a new answer or rule.\n{{ prompt }}",
        ))
        designer = DataDesigner(artifact_path=SYNTHETIC_DIR / "data_designer_artifacts", model_providers=[provider])
        designer.validate(builder)
        preview = designer.preview(config_builder=builder, num_records=len(seed_rows))
        dataset = preview.dataset
        if hasattr(dataset, "to_dict"):
            dataset = dataset.to_dict(orient="records")
        preview_records = dataset if isinstance(dataset, list) else []
        _write_jsonl(SYNTHETIC_DIR / "data_designer_preview.jsonl", preview_records)
        seed_by_prompt = {prompt_hash(row["prompt"]): row for row in accepted}
        screening_rows = []
        for record in preview_records:
            original = seed_by_prompt.get(prompt_hash(record.get("prompt", "")))
            prompt_variant = record.get("prompt_variant", "")
            variant_sample = copy.deepcopy(original) if original else None
            variant_verification = _screen_verification({**variant_sample, "prompt": prompt_variant}) if variant_sample and prompt_variant else {"answer_ok": False, "trace_ok": False}
            screening_rows.append({
                "task_type": record.get("task_type", ""),
                "original_prompt": record.get("prompt", ""),
                "prompt_variant": prompt_variant,
                "trace_variant": record.get("trace_variant", ""),
                "verified_against_original_solver": bool(variant_verification.get("answer_ok") and variant_verification.get("trace_ok")),
                "verification": json.dumps(variant_verification, ensure_ascii=False),
                "merge_status": "preview_only_never_ground_truth",
            })
        pd.DataFrame(screening_rows).to_csv(SYNTHETIC_DIR / "data_designer_preview_screening.csv", index=False)
        return {"status": "completed_local_vllm", "endpoint": endpoint, "served_model": served_model, "seed_counts": {task: 3 for task in ("gravity", "equation_numeric", "bit_manipulation")}, "records": len(preview_records), "verified_records": int(sum(row["verified_against_original_solver"] for row in screening_rows)), "merge_status": "preview_only_never_ground_truth"}
    except Exception as exc:
        return {"status": "blocked_runtime_error", "endpoint": endpoint, "served_model": served_model, "reason": repr(exc), "verified": False}


def _solve_candidate(sample: dict[str, object]) -> dict[str, object]:
    task_type = sample.get("task_type")
    if task_type == "gravity":
        result = solve_gravity_prompt(str(sample.get("prompt", "")))
        sample["answer"] = result.get("answer") or ""
        sample["trace"] = result.get("trace", {})
        sample["solver_status"] = result.get("status", "unknown")
        return result
    if task_type == "equation_numeric":
        result = solve_equation_numeric_prompt(str(sample.get("prompt", "")))
        outputs = result.get("query_outputs", [])
        sample["answer"] = outputs[0] if result.get("answer_unique") and outputs else ""
        sample["trace"] = {
            "solver": "24-rule-global-assignment",
            "query_operator": result.get("query_operator"),
            "query_outputs": outputs,
            "valid_assignment_count": result.get("valid_assignment_count", 0),
            "candidate_rules_by_operator": result.get("candidates_by_operator", {}),
            "selected_assignment": result.get("selected_assignment"),
        }
        sample["solver_status"] = result.get("status", "unknown")
        return result
    if task_type == "bit_manipulation":
        result = solve_bit_prompt(str(sample.get("prompt", "")))
        sample["answer"] = result.get("answer") or ""
        sample["trace"] = result.get("trace", {})
        sample["solver_status"] = result.get("status", "unknown")
        return result
    return {"status": "unsupported_task", "answer": None}


def _generation_plan(task_type: str, index: int) -> dict[str, object]:
    if task_type == "equation_numeric" and index >= 80:
        variant = "guess_answer_equivalent" if index < 90 else "guess_answer_ambiguous"
        return {"guess": True, "guess_variant": variant}
    return {"guess": False, "guess_variant": None}


def run_synthetic_pilot() -> dict[str, object]:
    rng = np.random.default_rng(SEED)
    generated, failures = [], []
    task_order = ["gravity", "equation_numeric", "bit_manipulation"]
    for task_type in task_order:
        for index in range(PILOT_CANDIDATES_PER_TASK):
            candidate_id = f"{task_type}-{index:04d}"
            plan = _generation_plan(task_type, index)
            try:
                if task_type == "gravity":
                    sample = generate_gravity_sample(rng)
                elif task_type == "equation_numeric":
                    sample = generate_equation_numeric_sample(rng, **plan)
                else:
                    sample = generate_bit_sample(rng)
                sample["candidate_id"] = candidate_id
                sample["candidate_index"] = candidate_id
                sample["requested_guess_variant"] = plan.get("guess_variant")
                _solve_candidate(sample)
                if task_type == "equation_numeric":
                    precheck = verify_equation_numeric_sample(sample)
                    sample["guess_subtype"] = precheck.get("guess_subtype", "pending")
                generated.append(sample)
            except Exception as exc:
                failure = {"candidate_id": candidate_id, "task_type": task_type, "generation_error": repr(exc)}
                failures.append(failure)
                generated.append({
                    "candidate_id": candidate_id, "candidate_index": candidate_id, "task_type": task_type,
                    "source": "deterministic_prior_generator", "generator_version": GENERATOR_VERSION,
                    "hidden_rule": {}, "prompt": "", "answer": "", "trace": {}, "difficulty": "generation_failed",
                    "deduce_or_guess": "unknown", "guess_subtype": "unknown", "generation_status": "generation_error",
                    "generation_error": repr(exc), "requested_guess_variant": plan.get("guess_variant"),
                })
    seen_exact, seen_normalized, seen_structure = set(), set(), set()
    screening, accepted, rejected = [], [], []
    for sample in generated:
        record = screen_synthetic_sample(sample, seen_exact, seen_normalized, seen_structure)
        screening.append(record)
        seen_exact.add(record["prompt_hash"])
        seen_normalized.add(record["normalized_prompt_hash"])
        seen_structure.add(record["structure_fingerprint"])
        sample["verification_status"] = record["verification_status"]
        sample["rejection_reasons"] = record["rejection_reasons"]
        (accepted if record["accepted"] else rejected).append(sample)
    _write_jsonl(SYNTHETIC_DIR / "synthetic_candidates.jsonl", generated)
    _write_jsonl(SYNTHETIC_DIR / "synthetic_accepted.jsonl", accepted)
    _write_jsonl(SYNTHETIC_DIR / "synthetic_rejected.jsonl", rejected)
    screening_df = pd.DataFrame(screening)
    screening_df.to_csv(SYNTHETIC_DIR / "synthetic_screening.csv", index=False)
    screening_df.to_csv(OUTPUT_DIR / "synthetic_pilot_screening.csv", index=False)
    cross_audit = _real_synthetic_cross_audit(generated)
    by_task = {}
    for task_type, group in screening_df.groupby("task_type", sort=True):
        reasons = Counter(reason for text in group["rejection_reasons"] for reason in str(text).split(";") if reason)
        by_task[task_type] = {"candidates": int(len(group)), "accepted": int(group["accepted"].sum()), "rejected": int((~group["accepted"]).sum()), "acceptance_rate": float(group["accepted"].mean()), "rejection_reasons": dict(reasons)}
    hidden_rule_counts, bit_operation_counts, gravity_values = Counter(), Counter(), []
    for sample in generated:
        hidden = sample.get("hidden_rule", {})
        if sample.get("task_type") == "equation_numeric": hidden_rule_counts.update(hidden.get("operator_to_rule", {}).values())
        elif sample.get("task_type") == "bit_manipulation": bit_operation_counts.update(hidden.get("operations", []))
        elif sample.get("task_type") == "gravity" and hidden.get("g") is not None: gravity_values.append(hidden["g"])
    data_designer_status = _data_designer_preview()
    metadata = {
        "schema_version": "nemotron-course-synthetic/v3", "seed": SEED, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "requested_candidates_per_task": PILOT_CANDIDATES_PER_TASK, "task_order": task_order, "generator_version": GENERATOR_VERSION,
        "generator_is_one_shot": True, "train_prior_only": True, "validation_or_holdout_used": False,
        "generation_failures": failures, "data_designer": data_designer_status,
        "data_designer_official_api": {"package": "data-designer", "model_provider": "ModelProvider(provider_type='openai', endpoint='http://127.0.0.1:8000/v1', api_key=None)", "seed_api": "LocalFileSeedSource + DataDesignerConfigBuilder.with_seed_dataset", "preview_api": "DataDesigner.preview(config_builder=..., num_records=...)"},
        "raw_candidate_artifact": "data/processed/synthetic/synthetic_candidates.jsonl",
    }
    (SYNTHETIC_DIR / "generation_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    guess_subtype_counts = {k: int(v) for k, v in screening_df["guess_subtype"].value_counts(dropna=False).to_dict().items()}
    statistics = {
        "mode": "pilot", "seed": SEED, "requested_candidates_per_task": PILOT_CANDIDATES_PER_TASK,
        "generated": len(generated), "accepted": len(accepted), "rejected": len(rejected), "generation_failures": len(failures),
        "by_task": by_task, "deduce_guess_counts": {k: int(v) for k, v in screening_df["deduce_or_guess"].value_counts(dropna=False).to_dict().items()},
        "deduce_guess_by_task": {task: {k: int(v) for k, v in group["deduce_or_guess"].value_counts(dropna=False).to_dict().items()} for task, group in screening_df.groupby("task_type", sort=True)},
        "guess_subtype_counts": guess_subtype_counts,
        "rule_coverage": {"equation_numeric": dict(hidden_rule_counts), "bit_operations": dict(bit_operation_counts), "gravity_g_unique": sorted(set(gravity_values))},
        "rejection_reasons": dict(Counter(reason for text in screening_df["rejection_reasons"] for reason in str(text).split(";") if reason)),
        "duplicate_audit": {"exact_duplicate_rows": int(len(screening_df) - screening_df["prompt_hash"].nunique()), "normalized_duplicate_rows": int(len(screening_df) - screening_df["normalized_prompt_hash"].nunique()), "structure_duplicate_rows": int(len(screening_df) - screening_df["structure_fingerprint"].nunique())},
        "real_cross_collection_audit": cross_audit,
        "data_designer": data_designer_status,
        "artifacts": {name: f"data/processed/synthetic/{name}" for name in ["synthetic_candidates.jsonl", "synthetic_accepted.jsonl", "synthetic_rejected.jsonl", "synthetic_screening.csv", "synthetic_real_leakage_audit.json", "synthetic_real_leakage_audit.csv", "generation_metadata.json"]},
    }
    (SYNTHETIC_DIR / "synthetic_statistics.json").write_text(json.dumps(statistics, ensure_ascii=False, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "synthetic_statistics.json").write_text(json.dumps(statistics, ensure_ascii=False, indent=2), encoding="utf-8")
    return statistics


if RUN_SYNTHETIC:
    synthetic_pilot_summary = run_synthetic_pilot()
else:
    existing_stats = SYNTHETIC_DIR / "synthetic_statistics.json"
    existing_accepted = SYNTHETIC_DIR / "synthetic_accepted.jsonl"
    if existing_stats.exists() and existing_accepted.exists():
        existing = json.loads(existing_stats.read_text(encoding="utf-8"))
        synthetic_pilot_summary = {
            "mode": "reused_existing_pilot",
            "reason": "RUN_SYNTHETIC=False；复用已生成的 300 条 Pilot，不重复生成。",
            "generated": existing.get("generated"),
            "accepted": existing.get("accepted"),
            "rejected": existing.get("rejected"),
            "statistics_path": str(existing_stats),
        }
    else:
        synthetic_pilot_summary = {"mode": "unavailable", "reason": "RUN_SYNTHETIC=False 且尚无已有 Pilot 产物。"}
print(json.dumps(synthetic_pilot_summary, ensure_ascii=False, indent=2))


## 8. 交付检查与未完成事项

本节汇总可复核文件和真实 Pilot 结果。Baseline 的准确率只有在 `RUN_BASELINE=True` 且本地 vLLM 实际完成后才会出现；默认仍关闭。当前 CPU 数据流程会生成 Gravity 分布审计、Operator 清单、单次候选记录、真实/合成交叉审计和 Verifier 负样本报告。Data Designer 使用本地 vLLM 的 Preview 是独立可选分支，不通过确定性 Verifier 的结果不会进入正式训练数据。


In [ ]:
expected_artifacts = [
    OUTPUT_DIR / "real_cleaned.csv",
    OUTPUT_DIR / "classification_audit.csv",
    OUTPUT_DIR / "dataset_statistics.json",
    OUTPUT_DIR / "split_manifest.json",
    SPLIT_DIR / "train.csv",
    SPLIT_DIR / "validation.csv",
    SPLIT_DIR / "holdout_test.csv",
    OUTPUT_DIR / "gravity_distribution_audit.json",
    OUTPUT_DIR / "gravity_distribution_comparison.png",
    OUTPUT_DIR / "equation_operator_inventory.json",
    OUTPUT_DIR / "synthetic_pilot_screening.csv",
    OUTPUT_DIR / "synthetic_statistics.json",
    SYNTHETIC_DIR / "synthetic_candidates.jsonl",
    SYNTHETIC_DIR / "synthetic_accepted.jsonl",
    SYNTHETIC_DIR / "synthetic_rejected.jsonl",
    SYNTHETIC_DIR / "synthetic_screening.csv",
    SYNTHETIC_DIR / "synthetic_real_leakage_audit.json",
    SYNTHETIC_DIR / "synthetic_real_leakage_audit.csv",
    SYNTHETIC_DIR / "generation_metadata.json",
]
missing = [str(path) for path in expected_artifacts if not path.exists()]
if missing:
    raise AssertionError(f"缺少产物：{missing}")
synthetic_screening_df = pd.read_csv(SYNTHETIC_DIR / "synthetic_screening.csv")
assert len(synthetic_screening_df) == 3 * PILOT_CANDIDATES_PER_TASK
assert synthetic_screening_df["verification_status"].notna().all()
assert set(synthetic_screening_df["task_type"]) == TARGET_TASKS
baseline_cpu_smoke_report = {
    "status": "passed",
    "run_baseline": RUN_BASELINE,
    "run_mode": RUN_MODE,
    "validation_samples_per_task": MAX_SAMPLES_PER_TASK,
    "baseline_status": baseline_status["status"],
    "verifier_version": OFFICIAL_VERIFIER_VERSION,
    "synthetic_mode": synthetic_pilot_summary.get("mode"),
    "synthetic_candidates": int(len(synthetic_screening_df)),
    "synthetic_accepted": int(synthetic_screening_df["accepted"].sum()),
    "synthetic_rejected": int((~synthetic_screening_df["accepted"]).sum()),
}
(OUTPUT_DIR / "baseline_cpu_smoke_report.json").write_text(json.dumps(baseline_cpu_smoke_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({
    "artifact_count": len(expected_artifacts),
    "missing": missing,
    "cleaned_rows": int(len(cleaned_df)),
    "baseline_status": baseline_status["status"],
    "synthetic_candidates": int(len(synthetic_screening_df)),
    "synthetic_accepted": int(synthetic_screening_df["accepted"].sum()),
    "synthetic_rejected": int((~synthetic_screening_df["accepted"]).sum()),
    "synthetic_by_task": synthetic_screening_df.groupby("task_type")["accepted"].agg(["count", "sum"]).to_dict(orient="index"),
    "rejection_reasons": synthetic_screening_df.loc[~synthetic_screening_df["accepted"]].groupby("task_type")["rejection_reasons"].apply(list).to_dict(),
    "guess_subtypes": synthetic_screening_df.loc[synthetic_screening_df["task_type"] == "equation_numeric", "guess_subtype"].value_counts().to_dict(),
    "data_designer_status": synthetic_pilot_summary.get("data_designer"),
}, ensure_ascii=False, indent=2))


In [ ]:
# Negative tests: each malformed candidate must be rejected by an independent Verifier or Screening.
from copy import deepcopy

candidate_rows = [json.loads(line) for line in (SYNTHETIC_DIR / "synthetic_candidates.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
accepted_by_task = {
    task: next((deepcopy(row) for row in candidate_rows if row.get("task_type") == task and row.get("verification_status") == "accepted"), None)
    for task in TARGET_TASKS
}
negative_results = []


def _negative_verifier_case(name: str, task_type: str, sample: dict[str, object], expected: str = "reject") -> None:
    result = _screen_verification(sample)
    rejected = not (result.get("answer_ok", False) and result.get("trace_ok", False) and result.get("identifiability", {}).get("answer_unique", False))
    negative_results.append({"case": name, "task_type": task_type, "expected": expected, "passed": bool(rejected), "result": result})


if accepted_by_task["gravity"]:
    gravity_case = deepcopy(accepted_by_task["gravity"])
    gravity_case["answer"] = "999999"
    _negative_verifier_case("gravity_wrong_answer", "gravity", gravity_case)
    gravity_case = deepcopy(accepted_by_task["gravity"])
    gravity_case["trace"]["query_t"] = "999"
    _negative_verifier_case("gravity_wrong_trace", "gravity", gravity_case)
    gravity_case["prompt"] = "Now, calculate the falling distance for t = 2s."
    _negative_verifier_case("gravity_unidentifiable_prompt", "gravity", gravity_case)
if accepted_by_task["equation_numeric"]:
    equation_case = deepcopy(accepted_by_task["equation_numeric"])
    equation_case["answer"] = "not-a-number"
    _negative_verifier_case("equation_wrong_answer", "equation_numeric", equation_case)
    equation_case = deepcopy(accepted_by_task["equation_numeric"])
    equation_case["trace"]["query_outputs"] = ["not-the-solver-output"]
    _negative_verifier_case("equation_wrong_trace", "equation_numeric", equation_case)
    equation_case["prompt"] = re.sub(r" = [^\n]+", " = invalid", equation_case["prompt"], count=1)
    _negative_verifier_case("equation_invalid_example_output", "equation_numeric", equation_case)
if accepted_by_task["bit_manipulation"]:
    bit_case = deepcopy(accepted_by_task["bit_manipulation"])
    bit_case["answer"] = "not-bits"
    _negative_verifier_case("bit_wrong_answer", "bit_manipulation", bit_case)
    bit_case = deepcopy(accepted_by_task["bit_manipulation"])
    bit_case["trace"]["expression"] = "AND(x)"
    _negative_verifier_case("bit_invalid_expression", "bit_manipulation", bit_case)
    bit_case["prompt"] = "\n".join(bit_case["prompt"].splitlines()[:2] + [bit_case["prompt"].splitlines()[-1]])
    _negative_verifier_case("bit_unidentifiable_prompt", "bit_manipulation", bit_case)

# Duplicate prompt must be rejected by Screening even when the Verifier accepts it.
if accepted_by_task["gravity"]:
    duplicate = deepcopy(accepted_by_task["gravity"])
    duplicate_record = screen_synthetic_sample(
        duplicate,
        {prompt_hash(duplicate["prompt"])},
        {synthetic_prompt_fingerprint(duplicate["prompt"])},
        {structure_fingerprint(duplicate["prompt"], duplicate["task_type"])},
    )
    negative_results.append({"case": "duplicate_prompt", "task_type": "gravity", "expected": "reject", "passed": bool(not duplicate_record["accepted"] and "duplicate_prompt" in duplicate_record["rejection_reasons"]), "result": duplicate_record})

negative_report = {
    "cases": negative_results,
    "passed": int(sum(case["passed"] for case in negative_results)),
    "total": len(negative_results),
    "all_passed": bool(negative_results) and all(case["passed"] for case in negative_results),
}
(OUTPUT_DIR / "verifier_negative_tests.json").write_text(json.dumps(negative_report, ensure_ascii=False, indent=2), encoding="utf-8")
if not negative_report["all_passed"]:
    raise AssertionError(negative_report)
print(json.dumps(negative_report, ensure_ascii=False, indent=2))
